In [1]:
# PROJET CLIENT 360
# Étape 1 : Vérification de l'environnement Python

import pandas as pd
import numpy as np

print("Bienvenue dans le projet CLIENT 360 !")
print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)

Bienvenue dans le projet CLIENT 360 !
Pandas : 3.0.3
NumPy : 2.4.6


In [2]:
# Vérifier le dossier dans lequel travaille Jupyter Notebook

import os

print("Dossier de travail actuel :")
print(os.getcwd())

Dossier de travail actuel :
C:\Users\Lenovo\Downloads\customer360


In [3]:
# Afficher les fichiers et dossiers disponibles

os.listdir()

['.ipynb_checkpoints',
 '.Rhistory',
 '01_Audit_Qualite_Donnees_clients.ipynb',
 'data',
 'documentation',
 'generate_customer360.py',
 'README.md']

In [4]:
# Afficher le contenu du dossier data

os.listdir("data")

['raw', 'reference']

In [5]:
# Afficher les fichiers de données brutes

os.listdir("data/raw")

['billing_customers.csv',
 'crm_customers.csv',
 'invoices.csv',
 'service_tickets.jsonl',
 'subscriptions.csv',
 'subscription_customers.csv']

In [6]:
# Importer les données brutes du CRM

import pandas as pd

df_crm = pd.read_csv("data/raw/crm_customers.csv")

# Afficher les cinq premières lignes
df_crm.head()

,prenom,nom,email,phone,address,city,province,postal_code,date_creation,crm_customer_id
0,Sophie,Lavoie,sophie.lavoie.1@demo.invalid,(438) 949-9547,7492 rue du Moulin,Longueuil,QC,H1V 0A3,2020-11-30,CRM-0000001
1,Amélie,Lavoie,amelie.lavoie.2@demo.invalid,(581) 356-5709,7364 rue des Érables,Lévis,QC,K6Z 9X0,2025-07-19,CRM-0000002
2,Samuel,Bélanger,samuel.belanger.3@exemple.ca,(581) 603-1098,7507 rue du Lac,Sherbrooke,QC,H8W 6Y3,2032-01-01,CRM-0000003
3,Camille,Pelletier,camille.pelletier.4@courriel.test,(438) 319-9611,8759 rue de la Gare,Gatineau,QC,G2H 4Y5,2021-05-15,CRM-0000004
4,Maxime,Pelletier,maxime.pelletier.5[at]courriel.test,(819) 759-1016,3855 rue des Pins,Laval,QC,H6Z 2C1,2022-11-05,CRM-0000005


In [7]:
# Afficher le nombre de lignes et de colonnes

print("Nombre de lignes :", df_crm.shape[0])
print("Nombre de colonnes :", df_crm.shape[1])

Nombre de lignes : 34422
Nombre de colonnes : 10


In [8]:
# Examiner la structure du fichier CRM

df_crm.info()

<class 'pandas.DataFrame'>
RangeIndex: 34422 entries, 0 to 34421
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   prenom           34422 non-null  str  
 1   nom              34422 non-null  str  
 2   email            32164 non-null  str  
 3   phone            31804 non-null  str  
 4   address          34422 non-null  str  
 5   city             34422 non-null  str  
 6   province         34422 non-null  str  
 7   postal_code      33231 non-null  str  
 8   date_creation    34422 non-null  str  
 9   crm_customer_id  34422 non-null  str  
dtypes: str(10)
memory usage: 6.3 MB


In [9]:
# Audit des valeurs manquantes

valeurs_manquantes = df_crm.isna().sum()

pourcentage_manquant = (
    df_crm.isna().mean() * 100
).round(2)

rapport_manquants = pd.DataFrame({
    "Valeurs manquantes": valeurs_manquantes,
    "Pourcentage (%)": pourcentage_manquant
})

rapport_manquants.sort_values(
    by="Valeurs manquantes",
    ascending=False
)

,Valeurs manquantes,Pourcentage (%)
phone,2618,7.61
email,2258,6.56
postal_code,1191,3.46
prenom,0,0.00
nom,0,0.00
address,0,0.00
city,0,0.00
province,0,0.00
date_creation,0,0.00
crm_customer_id,0,0.00


In [10]:
# Détection des doublons exacts

doublons_exacts = df_crm.duplicated().sum()

pourcentage_doublons = (
    doublons_exacts / len(df_crm) * 100
)

print("AUDIT DES DOUBLONS EXACTS")
print("-------------------------")

print("Nombre total d'enregistrements :", len(df_crm))

print("Nombre de doublons exacts :", doublons_exacts)

print(
    "Pourcentage de doublons exacts :",
    round(pourcentage_doublons, 2),
    "%"
)

AUDIT DES DOUBLONS EXACTS
-------------------------
Nombre total d'enregistrements : 34422
Nombre de doublons exacts : 0
Pourcentage de doublons exacts : 0.0 %


In [11]:
# Détection des doublons potentiels
# à partir du prénom et du nom

doublons_noms = df_crm.duplicated(
    subset=["prenom", "nom"],
    keep=False
)

print("AUDIT DES DOUBLONS POTENTIELS")
print("----------------------------")

print(
    "Nombre d'enregistrements partageant",
    "un même nom et prénom :",
    doublons_noms.sum()
)

AUDIT DES DOUBLONS POTENTIELS
----------------------------
Nombre d'enregistrements partageant un même nom et prénom : 33727


In [12]:
# Afficher les doublons potentiels

df_doublons_noms = df_crm.loc[
    doublons_noms,
    [
        "crm_customer_id",
        "prenom",
        "nom",
        "email",
        "phone",
        "city"
    ]
]

df_doublons_noms.sort_values(
    by=["nom", "prenom"]
).head(20)

,crm_customer_id,prenom,nom,email,phone,city
5405,CRM-0005406,Hugo,Bergero,hugo.bergeron.7948@exemple.ca,(581) 694-1125,Saguenay
11001,CRM-0011002,Hugo,Bergero,hugo.bergeron.16102@exemple.ca,(819) 357-8253,Québec
13724,CRM-0013725,Luc,Bergero,luc.bergeron.20064@courriel.test,(514) 973-7592,Drummondville
33234,CRM-0033235,Luc,Bergero,luc.bergeron.48277@courriel.test,(581) 848-9241,Saguenay
9511,CRM-0009512,Olivier,Bergero,olivier.bergeron.13992@demo.invalid,(450) 745-4309,Saguenay
34391,CRM-0034392,Olivier,Bergero,olivier.bergeron.49939@exemple.ca,(514) 205-1946,Saguenay
3494,CRM-0003495,ALEXANDRE,Bergeron,alexandre.bergeron.5148@courriel.test,(581) 337-8917,Longueuil
8674,CRM-0008675,ALEXANDRE,Bergeron,alexandre.bergeron.12713@exemple.ca,(819) 431-2991,Longueuil
8906,CRM-0008907,ALEXANDRE,Bergeron,alexandre.bergeron.13066@demo.invalid,(418) 635-8901,Trois-Rivières
15236,CRM-0015237,ALEXANDRE,Bergeron,alexandre.bergeron.22249@courriel.test,(438) 228-3103,Montréal


In [13]:
# Recherche des courriels identiques

# Exclure les courriels manquants
crm_avec_email = df_crm.dropna(subset=["email"]).copy()

# Identifier les courriels présents plusieurs fois
doublons_email = crm_avec_email.duplicated(
    subset=["email"],
    keep=False
)

print("AUDIT DES COURRIELS")
print("-------------------")

print(
    "Nombre d'enregistrements avec courriel :",
    len(crm_avec_email)
)

print(
    "Enregistrements partageant un même courriel :",
    doublons_email.sum()
)

# Afficher les correspondances
crm_avec_email.loc[
    doublons_email,
    ["crm_customer_id", "prenom", "nom", "email"]
].sort_values("email").head(20)

AUDIT DES COURRIELS
-------------------
Nombre d'enregistrements avec courriel : 32164
Enregistrements partageant un même courriel : 3866


,crm_customer_id,prenom,nom,email
14219,CRM-0014220,Alexandre,bélanger,alexandre.belanger.20788@demo.invalid
14220,CRM-0014221,Alexandre,Bélange,alexandre.belanger.20788@demo.invalid
30628,CRM-0030629,Alexandre,Bélanger,alexandre.belanger.44501@exemple.ca
30629,CRM-0030630,Alexandre,Bélanger,alexandre.belanger.44501@exemple.ca
8674,CRM-0008675,ALEXANDRE,Bergeron,alexandre.bergeron.12713@exemple.ca
8675,CRM-0008676,Alexandre,Bergeron,alexandre.bergeron.12713@exemple.ca
15889,CRM-0015890,Alexandre,Bergeron,alexandre.bergeron.23211@courriel.test
15890,CRM-0015891,ALEXANDRE,Bergeron,alexandre.bergeron.23211@courriel.test
19247,CRM-0019248,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca
19248,CRM-0019249,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca


In [14]:
df_crm.dropna(subset=["email"])

,prenom,nom,email,phone,address,city,province,postal_code,date_creation,crm_customer_id
0,Sophie,Lavoie,sophie.lavoie.1@demo.invalid,(438) 949-9547,7492 rue du Moulin,Longueuil,QC,H1V 0A3,2020-11-30,CRM-0000001
1,Amélie,Lavoie,amelie.lavoie.2@demo.invalid,(581) 356-5709,7364 rue des Érables,Lévis,QC,K6Z 9X0,2025-07-19,CRM-0000002
2,Samuel,Bélanger,samuel.belanger.3@exemple.ca,(581) 603-1098,7507 rue du Lac,Sherbrooke,QC,H8W 6Y3,2032-01-01,CRM-0000003
3,Camille,Pelletier,camille.pelletier.4@courriel.test,(438) 319-9611,8759 rue de la Gare,Gatineau,QC,G2H 4Y5,2021-05-15,CRM-0000004
4,Maxime,Pelletier,maxime.pelletier.5[at]courriel.test,(819) 759-1016,3855 rue des Pins,Laval,QC,H6Z 2C1,2022-11-05,CRM-0000005
...,...,...,...,...,...,...,...,...,...,...
34417,Olivier,Dubois,olivier.dubois.49983@demo.invalid,(819) 356-1872,526 rue Saint-Jean,Sherbrooke,QC,G1S 3S4,2024-07-03,CRM-0034418
34418,Fatima,Fortin,fatima.fortin.49988@exemple.ca,(418) 979-9736,5684 rue de la Gare,Saguenay,QC,L5M 8P6,2022-08-22,CRM-0034419
34419,Luc,Nguyen,luc.nguyen.49989@courriel.test,(450) 880-3772,8395 rue de l’Église,Sherbrooke,QC,J1W 1P5,2021-11-04,CRM-0034420
34420,OLIVIER,Gauthier,olivier.gauthier.49990@demo.invalid,(450) 782-1026,6559 rue des Érables,Drummondville,QC,G4W 0T3,2023-05-11,CRM-0034421


In [15]:
# Recherche des téléphones identiques

crm_avec_phone = df_crm.dropna(subset=["phone"]).copy()

doublons_phone = crm_avec_phone.duplicated(
    subset=["phone"],
    keep=False
)

print("AUDIT DES TÉLÉPHONES")
print("--------------------")

print(
    "Nombre d'enregistrements avec téléphone :",
    len(crm_avec_phone)
)

print(
    "Enregistrements partageant un même téléphone :",
    doublons_phone.sum()
)

crm_avec_phone.loc[
    doublons_phone,
    ["crm_customer_id", "prenom", "nom", "phone"]
].sort_values("phone").head(20)

AUDIT DES TÉLÉPHONES
--------------------
Nombre d'enregistrements avec téléphone : 31804
Enregistrements partageant un même téléphone : 3568


,crm_customer_id,prenom,nom,phone
33140,CRM-0033141,Hugo,Girard,(418) 204-5349
33141,CRM-0033142,Hugo,Girard,(418) 204-5349
13698,CRM-0013699,Léa,Lavoie,(418) 206-4342
13699,CRM-0013700,Léa,Lavoie,(418) 206-4342
9806,CRM-0009807,Maxime,Bergeron,(418) 207-3957
9807,CRM-0009808,MAXIME,Bergeron,(418) 207-3957
20063,CRM-0020064,Jean,Diallo,(418) 210-9496
20064,CRM-0020065,Jean,Diallo,(418) 210-9496
8986,CRM-0008987,Maxime,Bouchard,(418) 211-3617
8987,CRM-0008988,Maxime,bouchard,(418) 211-3617


In [16]:
# Recherche des correspondances
# sur le prénom, le nom et le téléphone

crm_identite_phone = df_crm.dropna(
    subset=["prenom", "nom", "phone"]
).copy()

doublons_identite_phone = crm_identite_phone.duplicated(
    subset=["prenom", "nom", "phone"],
    keep=False
)

print("AUDIT DES CORRESPONDANCES MULTICRITÈRES")
print("--------------------------------------")

print(
    "Enregistrements avec mêmes nom, prénom et téléphone :",
    doublons_identite_phone.sum()
)

crm_identite_phone.loc[
    doublons_identite_phone,
    [
        "crm_customer_id",
        "prenom",
        "nom",
        "phone",
        "email",
        "city"
    ]
].sort_values(
    by=["nom", "prenom", "phone"]
).head(20)

AUDIT DES CORRESPONDANCES MULTICRITÈRES
--------------------------------------
Enregistrements avec mêmes nom, prénom et téléphone : 2264


,crm_customer_id,prenom,nom,phone,email,city
19247,CRM-0019248,Alexandre,Bergeron,(438) 695-8150,alexandre.bergeron.28153@exemple.ca,Montréal
19248,CRM-0019249,Alexandre,Bergeron,(438) 695-8150,alexandre.bergeron.28153@exemple.ca,Montréal
31464,CRM-0031465,Alexandre,Bergeron,(514) 556-1938,alexandre.bergeron.45735@exemple.ca,Drummondville
31465,CRM-0031466,Alexandre,Bergeron,(514) 556-1938,alexandre.bergeron.45735@exemple.ca,Drummondville
9353,CRM-0009354,Amélie,Bergeron,(514) 609-6888,NaN,Lévis
9354,CRM-0009355,Amélie,Bergeron,(514) 609-6888,amelie.bergeron.13741@demo.invalid,Lévis
2319,CRM-0002320,Audrey,Bergeron,(438) 451-8147,audrey.bergeron.3415@courriel.test,Gatineau
2320,CRM-0002321,Audrey,Bergeron,(438) 451-8147,audrey.bergeron.3415@courriel.test,Gatineau
33053,CRM-0033054,Audrey,Bergeron,(514) 874-5571,audrey.bergeron.48020@courriel.test,Gatineau
33054,CRM-0033055,Audrey,Bergeron,(514) 874-5571,audrey.bergeron.48020@courriel.test,Gatineau


In [17]:
# Créer une copie de travail

df_crm_clean = df_crm.copy()

print("Nombre de lignes :", len(df_crm_clean))
print("Nombre de colonnes :", len(df_crm_clean.columns))

Nombre de lignes : 34422
Nombre de colonnes : 10


In [18]:
# Normalisation des noms et prénoms

import unicodedata

def normaliser_texte(texte):

    if pd.isna(texte):
        return None

    texte = str(texte).strip().lower()

    texte = unicodedata.normalize("NFKD", texte)

    texte = "".join(
        caractere
        for caractere in texte
        if not unicodedata.combining(caractere)
    )

    texte = " ".join(texte.split())

    return texte


df_crm_clean["prenom_normalise"] = (
    df_crm_clean["prenom"].apply(normaliser_texte)
)

df_crm_clean["nom_normalise"] = (
    df_crm_clean["nom"].apply(normaliser_texte)
)

df_crm_clean[
    ["prenom", "prenom_normalise", "nom", "nom_normalise"]
].head(20)

,prenom,prenom_normalise,nom,nom_normalise
0,Sophie,sophie,Lavoie,lavoie
1,Amélie,amelie,Lavoie,lavoie
2,Samuel,samuel,Bélanger,belanger
3,Camille,camille,Pelletier,pelletier
4,Maxime,maxime,Pelletier,pelletier
5,Hugo,hugo,Bergeron,bergeron
6,MICHEL,michel,Gagnon,gagnon
7,Michel,michel,Gagnon,gagnon
8,Olivier,olivier,Bouchard,bouchard
9,KARIM,karim,Leblanc,leblanc


In [19]:
# Normalisation des courriels

df_crm_clean["email_normalise"] = (
    df_crm_clean["email"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace("", pd.NA)
)

df_crm_clean[
    ["email", "email_normalise"]
].head(20)

,email,email_normalise
0,sophie.lavoie.1@demo.invalid,sophie.lavoie.1@demo.invalid
1,amelie.lavoie.2@demo.invalid,amelie.lavoie.2@demo.invalid
2,samuel.belanger.3@exemple.ca,samuel.belanger.3@exemple.ca
3,camille.pelletier.4@courriel.test,camille.pelletier.4@courriel.test
4,maxime.pelletier.5[at]courriel.test,maxime.pelletier.5[at]courriel.test
5,hugo.bergeron.7@exemple.ca,hugo.bergeron.7@exemple.ca
6,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test
7,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test
8,olivier.bouchard.9@demo.invalid,olivier.bouchard.9@demo.invalid
9,karim.leblanc.10@courriel.test,karim.leblanc.10@courriel.test


In [20]:
# Normalisation des téléphones

df_crm_clean["phone_normalise"] = (
    df_crm_clean["phone"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

df_crm_clean[
    ["phone", "phone_normalise"]
].head(20)

,phone,phone_normalise
0,(438) 949-9547,4389499547
1,(581) 356-5709,5813565709
2,(581) 603-1098,5816031098
3,(438) 319-9611,4383199611
4,(819) 759-1016,8197591016
5,(581) 716-4669,5817164669
6,(438) 580-8946,4385808946
7,(438) 580-8946,4385808946
8,(514) 913-6360,5149136360
9,(438) 222-8287,4382228287


In [21]:
# Comparaison avant et après normalisation

# AVANT : correspondances sur les noms originaux
avant = df_crm.duplicated(
    subset=["prenom", "nom"],
    keep=False
).sum()

# APRÈS : correspondances sur les noms normalisés
apres = df_crm_clean.duplicated(
    subset=["prenom_normalise", "nom_normalise"],
    keep=False
).sum()

# Calcul de la différence
difference = apres - avant

print("COMPARAISON AVANT / APRÈS NORMALISATION")
print("--------------------------------------")

print("Avant normalisation :", avant)
print("Après normalisation :", apres)
print("Correspondances supplémentaires :", difference)

COMPARAISON AVANT / APRÈS NORMALISATION
--------------------------------------
Avant normalisation : 33727
Après normalisation : 34144
Correspondances supplémentaires : 417


In [22]:
# Comparaison des courriels

# Avant normalisation
avant_email = df_crm["email"].dropna().duplicated(
    keep=False
).sum()

# Après normalisation
apres_email = df_crm_clean["email_normalise"].dropna().duplicated(
    keep=False
).sum()

print("COMPARAISON DES COURRIELS")
print("------------------------")

print("Avant normalisation :", avant_email)
print("Après normalisation :", apres_email)
print(
    "Correspondances supplémentaires :",
    apres_email - avant_email
)

COMPARAISON DES COURRIELS
------------------------
Avant normalisation : 3866
Après normalisation : 3866
Correspondances supplémentaires : 0


In [23]:
# Comparaison des téléphones

# Avant normalisation
avant_phone = df_crm["phone"].dropna().duplicated(
    keep=False
).sum()

# Après normalisation
apres_phone = df_crm_clean["phone_normalise"].dropna().duplicated(
    keep=False
).sum()

print("COMPARAISON DES TÉLÉPHONES")
print("-------------------------")

print("Avant normalisation :", avant_phone)
print("Après normalisation :", apres_phone)
print(
    "Correspondances supplémentaires :",
    apres_phone - avant_phone
)

COMPARAISON DES TÉLÉPHONES
-------------------------
Avant normalisation : 3568
Après normalisation : 4013
Correspondances supplémentaires : 445


In [24]:
# Validation du format des courriels

# Expression régulière pour vérifier le format général
regex_email = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

# Créer une colonne indiquant si le format est valide
df_crm_clean["email_valide"] = (
    df_crm_clean["email_normalise"]
    .str.fullmatch(regex_email)
    .fillna(False)
)

# Compter les résultats
emails_valides = df_crm_clean["email_valide"].sum()

emails_manquants = (
    df_crm_clean["email_normalise"].isna().sum()
)

emails_invalides = (
    (~df_crm_clean["email_valide"]
     & df_crm_clean["email_normalise"].notna()).sum()
)

print("AUDIT DE VALIDITÉ DES COURRIELS")
print("--------------------------------")
print("Courriels au format valide :", emails_valides)
print("Courriels au format invalide :", emails_invalides)
print("Courriels manquants :", emails_manquants)

AUDIT DE VALIDITÉ DES COURRIELS
--------------------------------
Courriels au format valide : 31403
Courriels au format invalide : 761
Courriels manquants : 2258


In [25]:
# Afficher les 20 premiers courriels invalides

df_crm_clean.loc[
    df_crm_clean["email_normalise"].notna()
    & ~df_crm_clean["email_valide"],
    ["crm_customer_id", "prenom", "nom", "email"]
].head(20)

,crm_customer_id,prenom,nom,email
4,CRM-0000005,Maxime,Pelletier,maxime.pelletier.5[at]courriel.test
57,CRM-0000058,Caroline,Lavoie,caroline.lavoie.79[at]demo.invalid
94,CRM-0000095,Alexandre,Diallo,alexandre.diallo.117[at]demo.invalid
175,CRM-0000176,Noémie,Kouam,noemie.kouam.230[at]exemple.ca
268,CRM-0000269,Léa,Tremblay,lea.tremblay.367[at]demo.invalid
314,CRM-0000315,Samuel,Pelletier,samuel.pelletier.432[at]demo.invalid
353,CRM-0000354,Caroline,Diallo,caroline.diallo.499[at]exemple.ca
412,CRM-0000413,CAROLINE,Bergeron,caroline.bergeron.589[at]courriel.test
493,CRM-0000494,Louis,lavoie,louis.lavoie.708[at]courriel.test
571,CRM-0000572,Nathalie,dufour,nathalie.dufour.826[at]demo.invalid


In [26]:
# Validation des numéros de téléphone

# Vérifier si le numéro contient exactement 10 chiffres
df_crm_clean["phone_valide"] = (
    df_crm_clean["phone_normalise"]
    .str.fullmatch(r"\d{10}")
    .fillna(False)
)

# Calcul des indicateurs
phones_valides = df_crm_clean["phone_valide"].sum()

phones_manquants = (
    df_crm_clean["phone_normalise"].isna().sum()
)

phones_invalides = (
    (~df_crm_clean["phone_valide"]
     & df_crm_clean["phone_normalise"].notna()).sum()
)

print("AUDIT DE VALIDITÉ DES TÉLÉPHONES")
print("--------------------------------")
print("Téléphones au format valide :", phones_valides)
print("Téléphones au format invalide :", phones_invalides)
print("Téléphones manquants :", phones_manquants)

AUDIT DE VALIDITÉ DES TÉLÉPHONES
--------------------------------
Téléphones au format valide : 31804
Téléphones au format invalide : 0
Téléphones manquants : 2618


In [27]:
# Afficher les 20 premiers téléphones invalides

df_crm_clean.loc[
    df_crm_clean["phone_normalise"].notna()
    & ~df_crm_clean["phone_valide"],
    ["crm_customer_id", "prenom", "nom", "phone", "phone_normalise"]
].head(20)

,crm_customer_id,prenom,nom,phone,phone_normalise


In [28]:
# Normalisation des codes postaux

df_crm_clean["postal_code_normalise"] = (
    df_crm_clean["postal_code"]
    .astype("string")
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False)
    .replace("", pd.NA)
)

# Vérification du format canadien
regex_postal = r"[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z]\d[ABCEGHJ-NPRSTV-Z]\d"

df_crm_clean["postal_code_valide"] = (
    df_crm_clean["postal_code_normalise"]
    .str.fullmatch(regex_postal)
    .fillna(False)
)

# Calcul des résultats
postaux_valides = df_crm_clean["postal_code_valide"].sum()

postaux_manquants = (
    df_crm_clean["postal_code_normalise"].isna().sum()
)

postaux_invalides = (
    (~df_crm_clean["postal_code_valide"]
     & df_crm_clean["postal_code_normalise"].notna()).sum()
)

print("AUDIT DES CODES POSTAUX")
print("-----------------------")
print("Codes postaux au format valide :", postaux_valides)
print("Codes postaux au format invalide :", postaux_invalides)
print("Codes postaux manquants :", postaux_manquants)

AUDIT DES CODES POSTAUX
-----------------------
Codes postaux au format valide : 33231
Codes postaux au format invalide : 0
Codes postaux manquants : 1191


In [29]:
# Correction des courriels mal formatés

# Conserver les courriels normalisés
df_crm_clean["email_corrige"] = (
    df_crm_clean["email_normalise"].copy()
)

# Remplacer [at] par @
df_crm_clean["email_corrige"] = (
    df_crm_clean["email_corrige"]
    .str.replace("[at]", "@", regex=False)
)

# Afficher quelques exemples
df_crm_clean[
    ["email", "email_normalise", "email_corrige"]
].head(20)

,email,email_normalise,email_corrige
0,sophie.lavoie.1@demo.invalid,sophie.lavoie.1@demo.invalid,sophie.lavoie.1@demo.invalid
1,amelie.lavoie.2@demo.invalid,amelie.lavoie.2@demo.invalid,amelie.lavoie.2@demo.invalid
2,samuel.belanger.3@exemple.ca,samuel.belanger.3@exemple.ca,samuel.belanger.3@exemple.ca
3,camille.pelletier.4@courriel.test,camille.pelletier.4@courriel.test,camille.pelletier.4@courriel.test
4,maxime.pelletier.5[at]courriel.test,maxime.pelletier.5[at]courriel.test,maxime.pelletier.5@courriel.test
5,hugo.bergeron.7@exemple.ca,hugo.bergeron.7@exemple.ca,hugo.bergeron.7@exemple.ca
6,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test
7,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test,michel.gagnon.8@courriel.test
8,olivier.bouchard.9@demo.invalid,olivier.bouchard.9@demo.invalid,olivier.bouchard.9@demo.invalid
9,karim.leblanc.10@courriel.test,karim.leblanc.10@courriel.test,karim.leblanc.10@courriel.test


In [30]:
# Vérification après correction

df_crm_clean["email_corrige_valide"] = (
    df_crm_clean["email_corrige"]
    .str.fullmatch(regex_email)
    .fillna(False)
)

# Calculer les indicateurs
valides_apres = df_crm_clean["email_corrige_valide"].sum()

invalides_apres = (
    (~df_crm_clean["email_corrige_valide"]
     & df_crm_clean["email_corrige"].notna()).sum()
)

print("VALIDATION DES COURRIELS APRÈS CORRECTION")
print("-----------------------------------------")

print("Courriels valides avant :", emails_valides)
print("Courriels valides après :", valides_apres)

print("Courriels invalides avant :", emails_invalides)
print("Courriels invalides après :", invalides_apres)

print(
    "Nombre de courriels devenus valides :",
    valides_apres - emails_valides
)

VALIDATION DES COURRIELS APRÈS CORRECTION
-----------------------------------------
Courriels valides avant : 31403
Courriels valides après : 32164
Courriels invalides avant : 761
Courriels invalides après : 0
Nombre de courriels devenus valides : 761


In [31]:
# Compter les courriels modifiés

courriels_modifies = (
    df_crm_clean["email_normalise"].notna()
    & (
        df_crm_clean["email_normalise"]
        != df_crm_clean["email_corrige"]
    )
)

print(
    "Nombre de courriels modifiés :",
    courriels_modifies.sum()
)

# Afficher quelques exemples
df_crm_clean.loc[
    courriels_modifies,
    [
        "crm_customer_id",
        "email_normalise",
        "email_corrige"
    ]
].head(20)

Nombre de courriels modifiés : 761


,crm_customer_id,email_normalise,email_corrige
4,CRM-0000005,maxime.pelletier.5[at]courriel.test,maxime.pelletier.5@courriel.test
57,CRM-0000058,caroline.lavoie.79[at]demo.invalid,caroline.lavoie.79@demo.invalid
94,CRM-0000095,alexandre.diallo.117[at]demo.invalid,alexandre.diallo.117@demo.invalid
175,CRM-0000176,noemie.kouam.230[at]exemple.ca,noemie.kouam.230@exemple.ca
268,CRM-0000269,lea.tremblay.367[at]demo.invalid,lea.tremblay.367@demo.invalid
314,CRM-0000315,samuel.pelletier.432[at]demo.invalid,samuel.pelletier.432@demo.invalid
353,CRM-0000354,caroline.diallo.499[at]exemple.ca,caroline.diallo.499@exemple.ca
412,CRM-0000413,caroline.bergeron.589[at]courriel.test,caroline.bergeron.589@courriel.test
493,CRM-0000494,louis.lavoie.708[at]courriel.test,louis.lavoie.708@courriel.test
571,CRM-0000572,nathalie.dufour.826[at]demo.invalid,nathalie.dufour.826@demo.invalid


In [32]:
# Journal des transformations

import pandas as pd
from datetime import datetime

journal_transformations = pd.DataFrame([
    {
        "date_traitement": datetime.now(),
        "source": "CRM",
        "colonne": "prenom / nom",
        "transformation": "Normalisation des accents, majuscules et espaces",
        "resultat": "417 correspondances supplémentaires",
        "statut": "Terminé"
    },
    {
        "date_traitement": datetime.now(),
        "source": "CRM",
        "colonne": "email",
        "transformation": "Normalisation et correction de [at] en @",
        "resultat": f"{courriels_modifies.sum()} courriels corrigés",
        "statut": "Terminé"
    },
    {
        "date_traitement": datetime.now(),
        "source": "CRM",
        "colonne": "phone",
        "transformation": "Suppression des caractères non numériques",
        "resultat": "445 correspondances supplémentaires",
        "statut": "Terminé"
    },
    {
        "date_traitement": datetime.now(),
        "source": "CRM",
        "colonne": "postal_code",
        "transformation": "Uniformisation du format des codes postaux",
        "resultat": "33 231 codes au format valide",
        "statut": "Terminé"
    }
])

journal_transformations

,date_traitement,source,colonne,transformation,resultat,statut
0,2026-11-14 11:34:41.059880,CRM,prenom / nom,"Normalisation des accents, majuscules et espaces",417 correspondances supplémentaires,Terminé
1,2026-11-14 11:34:41.059884,CRM,email,Normalisation et correction de [at] en @,761 courriels corrigés,Terminé
2,2026-11-14 11:34:41.060438,CRM,phone,Suppression des caractères non numériques,445 correspondances supplémentaires,Terminé
3,2026-11-14 11:34:41.060441,CRM,postal_code,Uniformisation du format des codes postaux,33 231 codes au format valide,Terminé


In [33]:
# Enregistrer le journal

from pathlib import Path

Path("documentation").mkdir(exist_ok=True)

journal_transformations.to_csv(
    "documentation/journal_transformations_crm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Journal des transformations enregistré avec succès.")

Journal des transformations enregistré avec succès.


In [34]:
# ÉTAPE  Rapport de qualité des données CRM

rapport_qualite = pd.DataFrame({
    "Indicateur": [
        "Nombre total d'enregistrements",
        "Courriels manquants",
        "Téléphones manquants",
        "Codes postaux manquants",
        "Courriels corrigés",
        "Courriels invalides après correction",
        "Téléphones au format invalide",
        "Codes postaux au format invalide"
    ],
    "Valeur": [
        len(df_crm_clean),
        df_crm_clean["email_corrige"].isna().sum(),
        df_crm_clean["phone_normalise"].isna().sum(),
        df_crm_clean["postal_code_normalise"].isna().sum(),
        courriels_modifies.sum(),
        invalides_apres,
        phones_invalides,
        postaux_invalides
    ]
})

rapport_qualite

,Indicateur,Valeur
0,Nombre total d'enregistrements,34422
1,Courriels manquants,2258
2,Téléphones manquants,2618
3,Codes postaux manquants,1191
4,Courriels corrigés,761
5,Courriels invalides après correction,0
6,Téléphones au format invalide,0
7,Codes postaux au format invalide,0


In [35]:
# Enregistrer le rapport de qualité

rapport_qualite.to_csv(
    "documentation/rapport_qualite_crm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Rapport de qualité CRM enregistré avec succès.")

Rapport de qualité CRM enregistré avec succès.


In [36]:
# Détection des correspondances fortes

# Définir les critères de rapprochement
criteres = [
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise"
]

# Exclure les enregistrements sans téléphone
df_identites = df_crm_clean.dropna(
    subset=criteres
).copy()

# Identifier les enregistrements partageant les mêmes critères
df_identites["correspondance_forte"] = (
    df_identites.duplicated(
        subset=criteres,
        keep=False
    )
)

# Extraire les correspondances
df_correspondances_fortes = df_identites.loc[
    df_identites["correspondance_forte"]
].copy()

print("AUDIT DES CORRESPONDANCES FORTES")
print("--------------------------------")

print(
    "Enregistrements concernés :",
    len(df_correspondances_fortes)
)

print(
    "Groupes de correspondances :",
    df_correspondances_fortes.groupby(
        criteres
    ).ngroups
)

AUDIT DES CORRESPONDANCES FORTES
--------------------------------
Enregistrements concernés : 3860
Groupes de correspondances : 1930


In [37]:
# Examiner les correspondances fortes

df_correspondances_fortes[
    [
        "crm_customer_id",
        "prenom",
        "nom",
        "email_corrige",
        "phone_normalise",
        "city"
    ]
].sort_values(
    by=["nom", "prenom", "phone_normalise"]
).head(30)

,crm_customer_id,prenom,nom,email_corrige,phone_normalise,city
15890,CRM-0015891,ALEXANDRE,Bergeron,alexandre.bergeron.23211@courriel.test,4389707005,Laval
8674,CRM-0008675,ALEXANDRE,Bergeron,alexandre.bergeron.12713@exemple.ca,8194312991,Longueuil
19025,CRM-0019026,AMÉLIE,Bergeron,amelie.bergeron.27839@courriel.test,4186343170,Drummondville
29972,CRM-0029973,AMÉLIE,Bergeron,amelie.bergeron.43551@exemple.ca,4509567650,Lévis
29203,CRM-0029204,AUDREY,Bergeron,audrey.bergeron.42453@exemple.ca,5147562385,Lévis
19247,CRM-0019248,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca,4386958150,Montréal
19248,CRM-0019249,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca,4386958150,Montréal
15889,CRM-0015890,Alexandre,Bergeron,alexandre.bergeron.23211@courriel.test,4389707005,Laval
31464,CRM-0031465,Alexandre,Bergeron,alexandre.bergeron.45735@exemple.ca,5145561938,Drummondville
31465,CRM-0031466,Alexandre,Bergeron,alexandre.bergeron.45735@exemple.ca,5145561938,Drummondville


In [38]:
# Correspondances sur les courriels corrigés

df_emails = df_crm_clean.loc[
    df_crm_clean["email_corrige_valide"]
].copy()

df_emails["correspondance_email"] = (
    df_emails.duplicated(
        subset=["email_corrige"],
        keep=False
    )
)

df_correspondances_email = df_emails.loc[
    df_emails["correspondance_email"]
].copy()

print("AUDIT DES CORRESPONDANCES PAR COURRIEL")
print("--------------------------------------")

print(
    "Enregistrements concernés :",
    len(df_correspondances_email)
)

print(
    "Groupes de correspondances :",
    df_correspondances_email.groupby(
        "email_corrige"
    ).ngroups
)

AUDIT DES CORRESPONDANCES PAR COURRIEL
--------------------------------------
Enregistrements concernés : 4076
Groupes de correspondances : 2038


In [39]:
# Détection des doublons à très forte confiance

# Définir les critères de rapprochement
criteres_tres_forts = [
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige"
]

# Conserver uniquement les enregistrements complets
# avec un courriel au format valide
df_tres_forts = df_crm_clean.loc[
    df_crm_clean[criteres_tres_forts].notna().all(axis=1)
    & df_crm_clean["email_corrige_valide"]
].copy()

# Identifier les correspondances sur les quatre critères
df_tres_forts["doublon_tres_fort"] = (
    df_tres_forts.duplicated(
        subset=criteres_tres_forts,
        keep=False
    )
)

# Extraire les enregistrements concernés
df_doublons_tres_forts = df_tres_forts.loc[
    df_tres_forts["doublon_tres_fort"]
].copy()

# Calculer le nombre de groupes
nombre_groupes = (
    df_doublons_tres_forts
    .groupby(criteres_tres_forts)
    .ngroups
)

print("AUDIT DES DOUBLONS À TRÈS FORTE CONFIANCE")
print("-----------------------------------------")

print(
    "Enregistrements concernés :",
    len(df_doublons_tres_forts)
)

print(
    "Groupes de doublons potentiels :",
    nombre_groupes
)

AUDIT DES DOUBLONS À TRÈS FORTE CONFIANCE
-----------------------------------------
Enregistrements concernés : 3386
Groupes de doublons potentiels : 1693


In [40]:
# Afficher les doublons à très forte confiance

df_doublons_tres_forts[
    [
        "crm_customer_id",
        "prenom",
        "nom",
        "email_corrige",
        "phone_normalise",
        "city"
    ]
].sort_values(
    by=[
        "nom",
        "prenom",
        "phone_normalise"
    ]
).head(30)

,crm_customer_id,prenom,nom,email_corrige,phone_normalise,city
15890,CRM-0015891,ALEXANDRE,Bergeron,alexandre.bergeron.23211@courriel.test,4389707005,Laval
8674,CRM-0008675,ALEXANDRE,Bergeron,alexandre.bergeron.12713@exemple.ca,8194312991,Longueuil
19025,CRM-0019026,AMÉLIE,Bergeron,amelie.bergeron.27839@courriel.test,4186343170,Drummondville
29972,CRM-0029973,AMÉLIE,Bergeron,amelie.bergeron.43551@exemple.ca,4509567650,Lévis
29203,CRM-0029204,AUDREY,Bergeron,audrey.bergeron.42453@exemple.ca,5147562385,Lévis
19247,CRM-0019248,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca,4386958150,Montréal
19248,CRM-0019249,Alexandre,Bergeron,alexandre.bergeron.28153@exemple.ca,4386958150,Montréal
15889,CRM-0015890,Alexandre,Bergeron,alexandre.bergeron.23211@courriel.test,4389707005,Laval
31464,CRM-0031465,Alexandre,Bergeron,alexandre.bergeron.45735@exemple.ca,5145561938,Drummondville
31465,CRM-0031466,Alexandre,Bergeron,alexandre.bergeron.45735@exemple.ca,5145561938,Drummondville


In [41]:
# Mesurer la redondance potentielle

nombre_enregistrements = len(df_doublons_tres_forts)

nombre_groupes = (
    df_doublons_tres_forts
    .groupby(criteres_tres_forts)
    .ngroups
)

redondance_potentielle = (
    nombre_enregistrements - nombre_groupes
)

taux_redondance = (
    redondance_potentielle / len(df_crm_clean)
) * 100

print("ANALYSE DE LA REDONDANCE POTENTIELLE")
print("-----------------------------------")

print("Enregistrements concernés :", nombre_enregistrements)
print("Groupes de correspondances :", nombre_groupes)

print(
    "Enregistrements potentiellement redondants :",
    redondance_potentielle
)

print(
    "Taux de redondance potentielle :",
    round(taux_redondance, 2),
    "%"
)

ANALYSE DE LA REDONDANCE POTENTIELLE
-----------------------------------
Enregistrements concernés : 3386
Groupes de correspondances : 1693
Enregistrements potentiellement redondants : 1693
Taux de redondance potentielle : 4.92 %


In [42]:
# Détection des conflits entre fiches clients

# Créer une copie des doublons à très forte confiance
df_conflits = df_doublons_tres_forts.copy()

print("ANALYSE DES CONFLITS ENTRE FICHES CLIENTS")
print("-----------------------------------------")

print("Nombre d'enregistrements analysés :", len(df_conflits))

print(
    "Nombre de groupes analysés :",
    df_conflits.groupby(criteres_tres_forts).ngroups
)

ANALYSE DES CONFLITS ENTRE FICHES CLIENTS
-----------------------------------------
Nombre d'enregistrements analysés : 3386
Nombre de groupes analysés : 1693


In [43]:
# Définir les colonnes à examiner
colonnes_conflits = [
    "address",
    "city",
    "postal_code_normalise"
]

# Vérifier les différences au sein de chaque groupe
analyse_conflits = (
    df_conflits
    .groupby(criteres_tres_forts, dropna=False)[colonnes_conflits]
    .nunique(dropna=True)
    .reset_index()
)

# Identifier les conflits
analyse_conflits["conflit_adresse"] = (
    analyse_conflits["address"] > 1
)

analyse_conflits["conflit_ville"] = (
    analyse_conflits["city"] > 1
)

analyse_conflits["conflit_code_postal"] = (
    analyse_conflits["postal_code_normalise"] > 1
)

# Déterminer les groupes présentant au moins un conflit
analyse_conflits["conflit_detecte"] = (
    analyse_conflits[
        [
            "conflit_adresse",
            "conflit_ville",
            "conflit_code_postal"
        ]
    ].any(axis=1)
)

# Afficher les résultats
analyse_conflits.head(10)

,prenom_normalise,nom_normalise,phone_normalise,email_corrige,address,city,postal_code_normalise,conflit_adresse,conflit_ville,conflit_code_postal,conflit_detecte
0,alexandre,belanger,4508447165,alexandre.belanger.44501@exemple.ca,1,1,1,False,False,False,False
1,alexandre,bergeron,4386958150,alexandre.bergeron.28153@exemple.ca,1,1,1,False,False,False,False
2,alexandre,bergeron,4389707005,alexandre.bergeron.23211@courriel.test,1,1,1,False,False,False,False
3,alexandre,bergeron,5145561938,alexandre.bergeron.45735@exemple.ca,1,1,1,False,False,False,False
4,alexandre,bergeron,8194312991,alexandre.bergeron.12713@exemple.ca,1,1,1,False,False,False,False
5,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,1,2,1,False,True,False,True
6,alexandre,bouchard,8194631539,alexandre.bouchard.7550@demo.invalid,1,1,1,False,False,False,False
7,alexandre,caron,4386834984,alexandre.caron.17212@courriel.test,1,1,1,False,False,False,False
8,alexandre,caron,5814936501,alexandre.caron.30535@exemple.ca,1,1,1,False,False,False,False
9,alexandre,cote,4505838134,alexandre.cote.27652@courriel.test,1,1,1,False,False,False,False


In [44]:
# Calculer les indicateurs de conflits

total_groupes = len(analyse_conflits)

groupes_conflits = analyse_conflits["conflit_detecte"].sum()

conflits_adresse = analyse_conflits["conflit_adresse"].sum()

conflits_ville = analyse_conflits["conflit_ville"].sum()

conflits_postal = analyse_conflits["conflit_code_postal"].sum()

taux_conflits = (
    groupes_conflits / total_groupes * 100
    if total_groupes > 0 else 0
)

print("RAPPORT DES CONFLITS ENTRE FICHES CLIENTS")
print("-----------------------------------------")

print("Groupes analysés :", total_groupes)

print("Groupes présentant un conflit :", groupes_conflits)

print("Conflits d'adresse :", conflits_adresse)

print("Conflits de ville :", conflits_ville)

print("Conflits de code postal :", conflits_postal)

print("Taux de groupes avec conflits :", round(taux_conflits, 2), "%")

RAPPORT DES CONFLITS ENTRE FICHES CLIENTS
-----------------------------------------
Groupes analysés : 1693
Groupes présentant un conflit : 56
Conflits d'adresse : 0
Conflits de ville : 56
Conflits de code postal : 0
Taux de groupes avec conflits : 3.31 %


In [45]:
# Analyse détaillée des conflits géographiques

# Extraire uniquement les groupes présentant un conflit
groupes_avec_conflits = analyse_conflits.loc[
    analyse_conflits["conflit_detecte"],
    criteres_tres_forts
].copy()

# Retrouver les fiches CRM appartenant à ces groupes
df_conflits_details = df_conflits.merge(
    groupes_avec_conflits,
    on=criteres_tres_forts,
    how="inner",
    validate="many_to_one"
)

print("ANALYSE DÉTAILLÉE DES CONFLITS")
print("--------------------------------")

print(
    "Nombre de groupes avec conflits :",
    len(groupes_avec_conflits)
)

print(
    "Nombre de fiches concernées :",
    len(df_conflits_details)
)

ANALYSE DÉTAILLÉE DES CONFLITS
--------------------------------
Nombre de groupes avec conflits : 56
Nombre de fiches concernées : 112


In [46]:
# Afficher les fiches présentant des conflits

colonnes_affichage = [
    "crm_customer_id",
    "prenom",
    "nom",
    "email_corrige",
    "phone_normalise",
    "address",
    "city",
    "postal_code_normalise"
]

df_conflits_details[
    colonnes_affichage
].sort_values(
    by=[
        "nom",
        "prenom",
        "phone_normalise"
    ]
).head(30)

,crm_customer_id,prenom,nom,email_corrige,phone_normalise,address,city,postal_code_normalise
38,CRM-0009814,Louis,Bergeron,louis.bergeron.14402@courriel.test,8194628280,568 rue du Parc,Trois-Rivieres,K3Y1K9
39,CRM-0009815,Louis,Bergeron,louis.bergeron.14402@courriel.test,8194628280,568 rue du Parc,Trois-Rivières,K3Y1K9
26,CRM-0007267,Alexandre,Bouchard,alexandre.bouchard.10706@demo.invalid,5145325447,5322 rue des Érables,Montréal,J7J8L1
27,CRM-0007268,Alexandre,Bouchard,alexandre.bouchard.10706@demo.invalid,5145325447,5322 rue des Érables,Montreal,J7J8L1
104,CRM-0031903,Karim,Bouchard,karim.bouchard.46357@courriel.test,8198045356,5565 rue du Moulin,Trois-Rivières,J2B8J3
28,CRM-0007516,LÉA,Bouchard,lea.bouchard.11084@courriel.test,8196708890,9621 rue du Moulin,Trois-Rivières,L5H5A2
29,CRM-0007517,LÉA,Bouchard,lea.bouchard.11084@courriel.test,8196708890,9621 rue du Moulin,Trois-Rivieres,<NA>
74,CRM-0025319,Olivier,Bouchard,olivier.bouchard.36863@exemple.ca,4188969564,3673 rue Saint-Jean,Quebec,G4N5L7
75,CRM-0025320,Olivier,Bouchard,olivier.bouchard.36863@exemple.ca,4188969564,3673 rue Saint-Jean,Québec,G4N5L7
60,CRM-0018546,Sarah,Bouchard,sarah.bouchard.27138@demo.invalid,4382649887,6926 rue de l’Église,Montréal,G9E1A8


In [47]:
# Analyser les villes différentes dans chaque groupe

comparaison_villes = (
    df_conflits_details
    .groupby(criteres_tres_forts)["city"]
    .agg(
        lambda x: sorted(
            set(x.dropna().astype(str))
        )
    )
    .reset_index(name="villes_identifiees")
)

# Afficher les groupes concernés
comparaison_villes.head(20)

,prenom_normalise,nom_normalise,phone_normalise,email_corrige,villes_identifiees
0,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,"[Montreal, Montréal]"
1,alexandre,girard,5815513567,alexandre.girard.39289@demo.invalid,"[Quebec, Québec]"
2,alexandre,morin,4182386704,alexandre.morin.40522@exemple.ca,"[Levis, Lévis]"
3,amelie,diallo,4385095624,amelie.diallo.9983@courriel.test,"[Levis, Lévis]"
4,amelie,levesque,8196777181,amelie.levesque.42695@demo.invalid,"[Levis, Lévis]"
5,audrey,gagne,5146119023,audrey.gagne.34730@exemple.ca,"[Levis, Lévis]"
6,audrey,levesque,5148788375,audrey.levesque.47233@exemple.ca,"[Levis, Lévis]"
7,audrey,pelletier,4386432801,audrey.pelletier.41462@demo.invalid,"[Montreal, Montréal]"
8,camille,caron,4383373684,camille.caron.38038@exemple.ca,"[Montreal, Montréal]"
9,camille,simard,4383451201,camille.simard.23434@courriel.test,"[Montreal, Montréal]"


In [48]:
# Examiner les villes contradictoires

# Afficher uniquement les villes identifiées
comparaison_villes[
    ["villes_identifiees"]
].head(30)

,villes_identifiees
0,"[Montreal, Montréal]"
1,"[Quebec, Québec]"
2,"[Levis, Lévis]"
3,"[Levis, Lévis]"
4,"[Levis, Lévis]"
5,"[Levis, Lévis]"
6,"[Levis, Lévis]"
7,"[Montreal, Montréal]"
8,"[Montreal, Montréal]"
9,"[Montreal, Montréal]"


In [49]:
# Normalisation des noms des villes

df_conflits_details["ville_normalisee"] = (
    df_conflits_details["city"]
    .apply(normaliser_texte)
)

# Comparaison avant et après normalisation
df_conflits_details[
    ["city", "ville_normalisee"]
].drop_duplicates().head(30)

,city,ville_normalisee
0,Montréal,montreal
1,Montreal,montreal
2,Lévis,levis
3,Levis,levis
10,Quebec,quebec
11,Québec,quebec
28,Trois-Rivières,trois-rivieres
29,Trois-Rivieres,trois-rivieres


In [50]:
# Recalculer les conflits de ville après normalisation

analyse_villes_normalisees = (
    df_conflits_details
    .groupby(criteres_tres_forts, dropna=False)
    ["ville_normalisee"]
    .nunique(dropna=True)
    .reset_index(name="nombre_villes_distinctes")
)

# Identifier les conflits persistants
analyse_villes_normalisees["conflit_reel_potentiel"] = (
    analyse_villes_normalisees["nombre_villes_distinctes"] > 1
)

# Calculer les résultats
conflits_persistants = (
    analyse_villes_normalisees["conflit_reel_potentiel"].sum()
)

conflits_resolus_normalisation = (
    len(analyse_villes_normalisees) - conflits_persistants
)

print("ANALYSE DES CONFLITS APRÈS NORMALISATION")
print("----------------------------------------")

print(
    "Groupes initialement en conflit :",
    len(analyse_villes_normalisees)
)

print(
    "Conflits résolus par normalisation :",
    conflits_resolus_normalisation
)

print(
    "Conflits persistants :",
    conflits_persistants
)

ANALYSE DES CONFLITS APRÈS NORMALISATION
----------------------------------------
Groupes initialement en conflit : 56
Conflits résolus par normalisation : 56
Conflits persistants : 0


In [51]:
# Vérification des dates de création

# Afficher quelques dates originales
print("EXEMPLES DE DATES DE CRÉATION")
print("----------------------------")

display(
    df_crm_clean[
        ["crm_customer_id", "date_creation"]
    ].head(15)
)

# Vérifier les valeurs manquantes
print(
    "Dates manquantes :",
    df_crm_clean["date_creation"].isna().sum()
)

EXEMPLES DE DATES DE CRÉATION
----------------------------


,crm_customer_id,date_creation
0,CRM-0000001,2020-11-30
1,CRM-0000002,2025-07-19
2,CRM-0000003,2032-01-01
3,CRM-0000004,2021-05-15
4,CRM-0000005,2022-11-05
5,CRM-0000006,2021-07-15
6,CRM-0000007,2023-09-23
7,CRM-0000008,2023-09-23
8,CRM-0000009,2023-07-07
9,CRM-0000010,2020-05-08


Dates manquantes : 0


In [52]:
# Conversion des dates de création

df_crm_clean["date_creation_dt"] = pd.to_datetime(
    df_crm_clean["date_creation"],
    errors="coerce"
)

print("AUDIT DES DATES DE CRÉATION")
print("--------------------------")

print(
    "Dates converties avec succès :",
    df_crm_clean["date_creation_dt"].notna().sum()
)

print(
    "Dates non converties :",
    df_crm_clean["date_creation_dt"].isna().sum()
)

display(
    df_crm_clean[
        ["crm_customer_id", "date_creation", "date_creation_dt"]
    ].head(15)
)

AUDIT DES DATES DE CRÉATION
--------------------------
Dates converties avec succès : 34422
Dates non converties : 0


,crm_customer_id,date_creation,date_creation_dt
0,CRM-0000001,2020-11-30,2020-11-30
1,CRM-0000002,2025-07-19,2025-07-19
2,CRM-0000003,2032-01-01,2032-01-01
3,CRM-0000004,2021-05-15,2021-05-15
4,CRM-0000005,2022-11-05,2022-11-05
5,CRM-0000006,2021-07-15,2021-07-15
6,CRM-0000007,2023-09-23,2023-09-23
7,CRM-0000008,2023-09-23,2023-09-23
8,CRM-0000009,2023-07-07,2023-07-07
9,CRM-0000010,2020-05-08,2020-05-08


In [53]:
# Détection des dates futures

import pandas as pd

# Définir la date de référence de l'audit
date_reference = pd.Timestamp("2026-11-15")

# Identifier les dates futures
df_crm_clean["date_future"] = (
    df_crm_clean["date_creation_dt"] > date_reference
)

# Compter les anomalies
nombre_dates_futures = df_crm_clean["date_future"].sum()

taux_dates_futures = (
    nombre_dates_futures / len(df_crm_clean) * 100
)

print("AUDIT DES DATES FUTURES")
print("-----------------------")

print("Nombre total de clients :", len(df_crm_clean))

print("Dates futures détectées :", nombre_dates_futures)

print(
    "Pourcentage de dates futures :",
    round(taux_dates_futures, 2),
    "%"
)

# Afficher les enregistrements concernés
display(
    df_crm_clean.loc[
        df_crm_clean["date_future"],
        ["crm_customer_id", "date_creation", "date_creation_dt"]
    ].head(20)
)

AUDIT DES DATES FUTURES
-----------------------
Nombre total de clients : 34422
Dates futures détectées : 414
Pourcentage de dates futures : 1.2 %


,crm_customer_id,date_creation,date_creation_dt
2,CRM-0000003,2032-01-01,2032-01-01
131,CRM-0000132,2032-01-01,2032-01-01
149,CRM-0000150,2032-01-01,2032-01-01
155,CRM-0000156,2032-01-01,2032-01-01
159,CRM-0000160,2032-01-01,2032-01-01
214,CRM-0000215,2032-01-01,2032-01-01
247,CRM-0000248,2032-01-01,2032-01-01
276,CRM-0000277,2032-01-01,2032-01-01
420,CRM-0000421,2032-01-01,2032-01-01
454,CRM-0000455,2032-01-01,2032-01-01


In [54]:
# Examiner les dates les plus anciennes et les plus récentes

print("PÉRIODE COUVERTE PAR LES DONNÉES CRM")
print("------------------------------------")

print(
    "Date de création la plus ancienne :",
    df_crm_clean["date_creation_dt"].min()
)

print(
    "Date de création la plus récente :",
    df_crm_clean["date_creation_dt"].max()
)

# Répartition des enregistrements par année
repartition_annees = (
    df_crm_clean["date_creation_dt"]
    .dt.year
    .value_counts()
    .sort_index()
)

display(repartition_annees)

PÉRIODE COUVERTE PAR LES DONNÉES CRM
------------------------------------
Date de création la plus ancienne : 2019-01-01 00:00:00
Date de création la plus récente : 2032-01-01 00:00:00


date_creation_dt
2019    4853
2020    4784
2021    4723
2022    4810
2023    4710
2024    4784
2025    4771
2026     573
2032     414
Name: count, dtype: int64

In [55]:
# Traitement des dates futures

# Conserver les dates originales
df_crm_clean["date_creation_fiable"] = (
    df_crm_clean["date_creation_dt"].copy()
)

# Mettre les dates futures à NaT
df_crm_clean.loc[
    df_crm_clean["date_future"],
    "date_creation_fiable"
] = pd.NaT

# Vérifier les résultats
print("TRAITEMENT DES DATES INCOHÉRENTES")
print("---------------------------------")

print(
    "Dates originales conservées :",
    df_crm_clean["date_creation_dt"].notna().sum()
)

print(
    "Dates fiables disponibles :",
    df_crm_clean["date_creation_fiable"].notna().sum()
)

print(
    "Dates futures exclues :",
    df_crm_clean["date_future"].sum()
)

# Afficher quelques exemples
display(
    df_crm_clean.loc[
        df_crm_clean["date_future"],
        [
            "crm_customer_id",
            "date_creation_dt",
            "date_creation_fiable"
        ]
    ].head(15)
)

TRAITEMENT DES DATES INCOHÉRENTES
---------------------------------
Dates originales conservées : 34422
Dates fiables disponibles : 34008
Dates futures exclues : 414


,crm_customer_id,date_creation_dt,date_creation_fiable
2,CRM-0000003,2032-01-01,NaT
131,CRM-0000132,2032-01-01,NaT
149,CRM-0000150,2032-01-01,NaT
155,CRM-0000156,2032-01-01,NaT
159,CRM-0000160,2032-01-01,NaT
214,CRM-0000215,2032-01-01,NaT
247,CRM-0000248,2032-01-01,NaT
276,CRM-0000277,2032-01-01,NaT
420,CRM-0000421,2032-01-01,NaT
454,CRM-0000455,2032-01-01,NaT


In [56]:
# Examiner les différentes dates futures

print("DATES FUTURES IDENTIFIÉES")
print("-------------------------")

dates_futures_distinctes = (
    df_crm_clean.loc[
        df_crm_clean["date_future"],
        "date_creation_dt"
    ]
    .value_counts()
    .sort_index()
)

display(dates_futures_distinctes)

DATES FUTURES IDENTIFIÉES
-------------------------


date_creation_dt
2032-01-01    414
Name: count, dtype: int64

In [57]:
# Calcul du score de qualité des fiches CRM

# Vérifier les courriels corrigés
email_ok = (
    df_crm_clean["email_corrige_valide"]
    .fillna(False)
)

# Vérifier les téléphones
phone_ok = (
    df_crm_clean["phone_valide"]
    .fillna(False)
)

# Vérifier les codes postaux
postal_ok = (
    df_crm_clean["postal_code_valide"]
    .fillna(False)
)

# Vérifier les dates fiables
date_ok = (
    df_crm_clean["date_creation_fiable"]
    .notna()
)

# Vérifier les noms et prénoms
identite_ok = (
    df_crm_clean["prenom_normalise"].notna()
    & df_crm_clean["nom_normalise"].notna()
    & df_crm_clean["prenom_normalise"].ne("")
    & df_crm_clean["nom_normalise"].ne("")
)

# Calculer le score sur 100 points
df_crm_clean["score_qualite"] = (
    email_ok.astype(int) * 25
    + phone_ok.astype(int) * 25
    + postal_ok.astype(int) * 20
    + date_ok.astype(int) * 20
    + identite_ok.astype(int) * 10
)

print("CALCUL DU SCORE DE QUALITÉ")
print("--------------------------")

print(
    "Nombre de fiches évaluées :",
    len(df_crm_clean)
)

print(
    "Score moyen :",
    round(df_crm_clean["score_qualite"].mean(), 2)
)

print(
    "Score minimum :",
    df_crm_clean["score_qualite"].min()
)

print(
    "Score maximum :",
    df_crm_clean["score_qualite"].max()
)

# Afficher quelques résultats
display(
    df_crm_clean[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite"
        ]
    ].head(20)
)

CALCUL DU SCORE DE QUALITÉ
--------------------------
Nombre de fiches évaluées : 34422
Score moyen : 95.53
Score minimum : 30
Score maximum : 100


,crm_customer_id,prenom,nom,score_qualite
0,CRM-0000001,Sophie,Lavoie,100
1,CRM-0000002,Amélie,Lavoie,100
2,CRM-0000003,Samuel,Bélanger,80
3,CRM-0000004,Camille,Pelletier,100
4,CRM-0000005,Maxime,Pelletier,100
5,CRM-0000006,Hugo,Bergeron,100
6,CRM-0000007,MICHEL,Gagnon,100
7,CRM-0000008,Michel,Gagnon,100
8,CRM-0000009,Olivier,Bouchard,100
9,CRM-0000010,KARIM,Leblanc,100


In [58]:
# Distribution des scores de qualité

print("DISTRIBUTION DES SCORES DE QUALITÉ")
print("----------------------------------")

distribution_scores = (
    df_crm_clean["score_qualite"]
    .value_counts()
    .sort_index(ascending=False)
)

display(distribution_scores)

# Calculer les indicateurs globaux
print(
    "Fiches avec un score de 100 :",
    (df_crm_clean["score_qualite"] == 100).sum()
)

print(
    "Fiches avec un score inférieur à 70 :",
    (df_crm_clean["score_qualite"] < 70).sum()
)

print(
    "Pourcentage de fiches avec un score de 100 :",
    round(
        (df_crm_clean["score_qualite"] == 100).mean() * 100,
        2
    ),
    "%"
)

DISTRIBUTION DES SCORES DE QUALITÉ
----------------------------------


score_qualite
100    28329
80      1373
75      4343
60         8
55       205
50       153
30        11
Name: count, dtype: int64

Fiches avec un score de 100 : 28329
Fiches avec un score inférieur à 70 : 377
Pourcentage de fiches avec un score de 100 : 82.3 %


In [61]:
# Comparaison des scores des doublons

# Définir les critères de rapprochement
criteres_tres_forts = [
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige"
]

# Récupérer les doublons à très forte confiance
# et ajouter leurs scores de qualité

df_comparaison_scores = df_doublons_tres_forts.drop(
    columns=["score_qualite"],
    errors="ignore"
).merge(
    df_crm_clean[
        ["crm_customer_id", "score_qualite"]
    ],
    on="crm_customer_id",
    how="left",
    validate="one_to_one"
)

# Vérifier les résultats
print("COMPARAISON DES SCORES DE QUALITÉ")
print("--------------------------------")

print(
    "Nombre de fiches concernées :",
    len(df_comparaison_scores)
)

print(
    "Nombre de groupes :",
    df_comparaison_scores.groupby(
        criteres_tres_forts
    ).ngroups
)

print(
    "Scores manquants :",
    df_comparaison_scores["score_qualite"].isna().sum()
)

# Afficher les fiches avec leurs scores
display(
    df_comparaison_scores[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "email_corrige",
            "phone_normalise",
            "score_qualite"
        ]
    ].head(20)
)

COMPARAISON DES SCORES DE QUALITÉ
--------------------------------
Nombre de fiches concernées : 3386
Nombre de groupes : 1693
Scores manquants : 0


,crm_customer_id,prenom,nom,email_corrige,phone_normalise,score_qualite
0,CRM-0000007,MICHEL,Gagnon,michel.gagnon.8@courriel.test,4385808946,100
1,CRM-0000008,Michel,Gagnon,michel.gagnon.8@courriel.test,4385808946,100
2,CRM-0000017,Isabelle,Roy,isabelle.roy.19@exemple.ca,5143376750,100
3,CRM-0000018,Isabelle,Roy,isabelle.roy.19@exemple.ca,5143376750,100
4,CRM-0000068,David,Bergeron,david.bergeron.89@courriel.test,8194647924,100
5,CRM-0000069,David,bergeron,david.bergeron.89@courriel.test,8194647924,100
6,CRM-0000084,Caroline,bouchard,caroline.bouchard.105@exemple.ca,4389864506,80
7,CRM-0000085,Caroline,Bouchard,caroline.bouchard.105@exemple.ca,4389864506,100
8,CRM-0000088,Audrey,Kouam,audrey.kouam.108@exemple.ca,4382686816,100
9,CRM-0000089,AUDREY,Kouam,audrey.kouam.108@exemple.ca,4382686816,100


In [62]:
# Comparaison des scores au sein des groupes

# Calculer les scores minimum et maximum par groupe
comparaison_groupes = (
    df_comparaison_scores
    .groupby(criteres_tres_forts)
    .agg(
        nombre_fiches=("crm_customer_id", "count"),
        score_min=("score_qualite", "min"),
        score_max=("score_qualite", "max")
    )
    .reset_index()
)

# Calculer l'écart entre les scores
comparaison_groupes["ecart_score"] = (
    comparaison_groupes["score_max"]
    - comparaison_groupes["score_min"]
)

# Identifier les groupes ayant des scores différents
comparaison_groupes["qualite_differente"] = (
    comparaison_groupes["ecart_score"] > 0
)

# Afficher les résultats
print("ANALYSE DES DIFFÉRENCES DE QUALITÉ")
print("---------------------------------")

print(
    "Nombre total de groupes :",
    len(comparaison_groupes)
)

print(
    "Groupes avec des scores différents :",
    comparaison_groupes["qualite_differente"].sum()
)

print(
    "Groupes avec des scores identiques :",
    (~comparaison_groupes["qualite_differente"]).sum()
)

# Afficher les 20 premiers groupes
display(comparaison_groupes.head(20))

ANALYSE DES DIFFÉRENCES DE QUALITÉ
---------------------------------
Nombre total de groupes : 1693
Groupes avec des scores différents : 147
Groupes avec des scores identiques : 1546


,prenom_normalise,nom_normalise,phone_normalise,email_corrige,nombre_fiches,score_min,score_max,ecart_score,qualite_differente
0,alexandre,belanger,4508447165,alexandre.belanger.44501@exemple.ca,2,80,100,20,True
1,alexandre,bergeron,4386958150,alexandre.bergeron.28153@exemple.ca,2,100,100,0,False
2,alexandre,bergeron,4389707005,alexandre.bergeron.23211@courriel.test,2,100,100,0,False
3,alexandre,bergeron,5145561938,alexandre.bergeron.45735@exemple.ca,2,100,100,0,False
4,alexandre,bergeron,8194312991,alexandre.bergeron.12713@exemple.ca,2,100,100,0,False
5,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,2,100,100,0,False
6,alexandre,bouchard,8194631539,alexandre.bouchard.7550@demo.invalid,2,100,100,0,False
7,alexandre,caron,4386834984,alexandre.caron.17212@courriel.test,2,100,100,0,False
8,alexandre,caron,5814936501,alexandre.caron.30535@exemple.ca,2,100,100,0,False
9,alexandre,cote,4505838134,alexandre.cote.27652@courriel.test,2,100,100,0,False


In [63]:
# Classement des fiches candidates principales

# Créer une copie de travail
df_selection = df_comparaison_scores.copy()

# Récupérer les dates fiables depuis le CRM nettoyé
df_selection = df_selection.drop(
    columns=["date_creation_fiable"],
    errors="ignore"
).merge(
    df_crm_clean[
        ["crm_customer_id", "date_creation_fiable"]
    ],
    on="crm_customer_id",
    how="left",
    validate="one_to_one"
)

# Classer les fiches :
# 1. Par groupe de correspondance
# 2. Par score de qualité décroissant
# 3. Par date de création croissante
# 4. Par identifiant CRM pour départager les égalités

df_selection = df_selection.sort_values(
    by=criteres_tres_forts + [
        "score_qualite",
        "date_creation_fiable",
        "crm_customer_id"
    ],
    ascending=[True, True, True, True,
               False, True, True],
    na_position="last"
)

# Attribuer un rang à chaque fiche
df_selection["rang_fiche"] = (
    df_selection
    .groupby(criteres_tres_forts)
    .cumcount() + 1
)

# Identifier les fiches candidates principales
df_selection["candidate_principale"] = (
    df_selection["rang_fiche"] == 1
)

# Afficher les résultats
print("CLASSEMENT DES FICHES CLIENTS")
print("-----------------------------")

print(
    "Nombre de fiches analysées :",
    len(df_selection)
)

print(
    "Fiches candidates principales :",
    df_selection["candidate_principale"].sum()
)

print(
    "Autres fiches des groupes :",
    (~df_selection["candidate_principale"]).sum()
)

display(
    df_selection[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite",
            "date_creation_fiable",
            "rang_fiche",
            "candidate_principale"
        ]
    ].head(20)
)

CLASSEMENT DES FICHES CLIENTS
-----------------------------
Nombre de fiches analysées : 3386
Fiches candidates principales : 1693
Autres fiches des groupes : 1693


,crm_customer_id,prenom,nom,score_qualite,date_creation_fiable,rang_fiche,candidate_principale
3051,CRM-0030630,Alexandre,Bélanger,100,2020-04-12,1,True
3050,CRM-0030629,Alexandre,Bélanger,80,NaT,2,False
1898,CRM-0019248,Alexandre,Bergeron,100,2019-04-23,1,True
1899,CRM-0019249,Alexandre,Bergeron,100,2019-04-23,2,False
1550,CRM-0015890,Alexandre,Bergeron,100,2022-07-15,1,True
1551,CRM-0015891,ALEXANDRE,Bergeron,100,2022-07-15,2,False
3124,CRM-0031465,Alexandre,Bergeron,100,2024-03-23,1,True
3125,CRM-0031466,Alexandre,Bergeron,100,2024-03-23,2,False
862,CRM-0008675,ALEXANDRE,Bergeron,100,2023-02-07,1,True
863,CRM-0008676,Alexandre,Bergeron,100,2023-02-07,2,False


In [64]:
# Vérification de la sélection des fiches principales

# Compter les fiches principales par groupe
verification_selection = (
    df_selection
    .groupby(criteres_tres_forts)
    .agg(
        nombre_fiches=("crm_customer_id", "count"),
        nombre_principales=("candidate_principale", "sum")
    )
    .reset_index()
)

# Identifier les groupes correctement sélectionnés
verification_selection["selection_valide"] = (
    verification_selection["nombre_principales"] == 1
)

# Calculer les indicateurs
total_groupes = len(verification_selection)

groupes_valides = (
    verification_selection["selection_valide"].sum()
)

groupes_invalides = (
    (~verification_selection["selection_valide"]).sum()
)

# Afficher les résultats
print("AUDIT DE LA SÉLECTION DES FICHES PRINCIPALES")
print("--------------------------------------------")

print("Nombre total de groupes :", total_groupes)

print("Groupes avec une seule fiche principale :", groupes_valides)

print("Groupes avec une sélection incorrecte :", groupes_invalides)

print(
    "Taux de sélection valide :",
    round(groupes_valides / total_groupes * 100, 2)
    if total_groupes > 0 else 0,
    "%"
)

# Afficher les premières lignes
display(verification_selection.head(20))

AUDIT DE LA SÉLECTION DES FICHES PRINCIPALES
--------------------------------------------
Nombre total de groupes : 1693
Groupes avec une seule fiche principale : 1693
Groupes avec une sélection incorrecte : 0
Taux de sélection valide : 100.0 %


,prenom_normalise,nom_normalise,phone_normalise,email_corrige,nombre_fiches,nombre_principales,selection_valide
0,alexandre,belanger,4508447165,alexandre.belanger.44501@exemple.ca,2,1,True
1,alexandre,bergeron,4386958150,alexandre.bergeron.28153@exemple.ca,2,1,True
2,alexandre,bergeron,4389707005,alexandre.bergeron.23211@courriel.test,2,1,True
3,alexandre,bergeron,5145561938,alexandre.bergeron.45735@exemple.ca,2,1,True
4,alexandre,bergeron,8194312991,alexandre.bergeron.12713@exemple.ca,2,1,True
5,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,2,1,True
6,alexandre,bouchard,8194631539,alexandre.bouchard.7550@demo.invalid,2,1,True
7,alexandre,caron,4386834984,alexandre.caron.17212@courriel.test,2,1,True
8,alexandre,caron,5814936501,alexandre.caron.30535@exemple.ca,2,1,True
9,alexandre,cote,4505838134,alexandre.cote.27652@courriel.test,2,1,True


In [65]:
# Analyse des égalités de scores et de dates

# Récupérer le score maximal de chaque groupe
df_selection["score_max_groupe"] = (
    df_selection
    .groupby(criteres_tres_forts)["score_qualite"]
    .transform("max")
)

# Identifier les fiches ayant le meilleur score
df_selection["score_maximal"] = (
    df_selection["score_qualite"]
    == df_selection["score_max_groupe"]
)

# Compter les fiches partageant le score maximal
egalites_scores = (
    df_selection
    .groupby(criteres_tres_forts)["score_maximal"]
    .sum()
)

groupes_egalite_score = (egalites_scores > 1).sum()

# Examiner les dates des fiches ayant le score maximal
df_meilleurs_scores = df_selection.loc[
    df_selection["score_maximal"]
].copy()

# Compter les groupes où les meilleures fiches
# partagent également la même date la plus ancienne
def egalite_date_groupe(groupe):

    if len(groupe) < 2:
        return False

    dates = groupe["date_creation_fiable"]

    if dates.notna().any():
        date_min = dates.min()
        return (dates == date_min).sum() > 1

    # Si toutes les dates sont manquantes,
    # le critère de date ne départage pas les fiches.
    return True


egalites_dates = (
    df_meilleurs_scores
    .groupby(criteres_tres_forts)
    .apply(egalite_date_groupe)
)

groupes_egalite_score_date = egalites_dates.sum()

# Afficher les résultats
print("AUDIT DES ÉGALITÉS ENTRE FICHES CLIENTS")
print("--------------------------------------")

print("Nombre total de groupes :", len(egalites_scores))

print(
    "Groupes avec égalité du meilleur score :",
    groupes_egalite_score
)

print(
    "Groupes avec égalité du meilleur score et de la date :",
    groupes_egalite_score_date
)

print(
    "Groupes départagés uniquement par l'identifiant CRM :",
    groupes_egalite_score_date
)

AUDIT DES ÉGALITÉS ENTRE FICHES CLIENTS
--------------------------------------
Nombre total de groupes : 1693
Groupes avec égalité du meilleur score : 1546
Groupes avec égalité du meilleur score et de la date : 1543
Groupes départagés uniquement par l'identifiant CRM : 1543


In [66]:
# Identifier les groupes départagés par l'identifiant CRM

# Récupérer les groupes présentant une égalité
groupes_a_departager = (
    egalites_dates[egalites_dates]
    .reset_index()
)

# Conserver uniquement les critères de rapprochement
groupes_a_departager = groupes_a_departager[
    criteres_tres_forts
]

# Récupérer les fiches appartenant à ces groupes
df_departage = df_selection.merge(
    groupes_a_departager,
    on=criteres_tres_forts,
    how="inner",
    validate="many_to_one"
)

# Conserver uniquement les fiches ayant le score maximal
df_departage = df_departage.loc[
    df_departage["score_maximal"]
].copy()

# Identifier la meilleure date dans chaque groupe
df_departage["date_min_groupe"] = (
    df_departage
    .groupby(criteres_tres_forts)["date_creation_fiable"]
    .transform("min")
)

# Conserver les fiches ayant la date prioritaire
# Si toutes les dates sont manquantes, conserver toutes les candidates
df_departage = df_departage.loc[
    (
        df_departage["date_creation_fiable"]
        == df_departage["date_min_groupe"]
    )
    |
    df_departage["date_min_groupe"].isna()
].copy()

# Vérifier les résultats
print("AUDIT DES GROUPES NÉCESSITANT UN DÉPARTAGE")
print("-----------------------------------------")

print(
    "Nombre de groupes concernés :",
    df_departage.groupby(criteres_tres_forts).ngroups
)

print(
    "Nombre de fiches candidates concernées :",
    len(df_departage)
)

# Afficher quelques exemples
display(
    df_departage[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite",
            "date_creation_fiable",
            "rang_fiche",
            "candidate_principale"
        ]
    ].head(20)
)

AUDIT DES GROUPES NÉCESSITANT UN DÉPARTAGE
-----------------------------------------
Nombre de groupes concernés : 1543
Nombre de fiches candidates concernées : 3086


,crm_customer_id,prenom,nom,score_qualite,date_creation_fiable,rang_fiche,candidate_principale
0,CRM-0019248,Alexandre,Bergeron,100,2019-04-23,1,True
1,CRM-0019249,Alexandre,Bergeron,100,2019-04-23,2,False
2,CRM-0015890,Alexandre,Bergeron,100,2022-07-15,1,True
3,CRM-0015891,ALEXANDRE,Bergeron,100,2022-07-15,2,False
4,CRM-0031465,Alexandre,Bergeron,100,2024-03-23,1,True
5,CRM-0031466,Alexandre,Bergeron,100,2024-03-23,2,False
6,CRM-0008675,ALEXANDRE,Bergeron,100,2023-02-07,1,True
7,CRM-0008676,Alexandre,Bergeron,100,2023-02-07,2,False
8,CRM-0007267,Alexandre,Bouchard,100,2024-06-25,1,True
9,CRM-0007268,Alexandre,Bouchard,100,2024-06-25,2,False


In [67]:
# Création du journal des décisions de sélection

# Récupérer uniquement les fiches principales
df_decisions = df_selection.loc[
    df_selection["candidate_principale"]
].copy()

# Préparer les informations de départage
df_decisions = df_decisions.merge(
    egalites_scores.rename("nombre_scores_max").reset_index(),
    on=criteres_tres_forts,
    how="left",
    validate="one_to_one"
)

df_decisions = df_decisions.merge(
    egalites_dates.rename("egalite_score_date").reset_index(),
    on=criteres_tres_forts,
    how="left",
    validate="one_to_one"
)

# Déterminer le critère ayant permis la sélection
def determiner_critere_selection(ligne):

    if ligne["nombre_scores_max"] == 1:
        return "Meilleur score de qualité"

    elif not ligne["egalite_score_date"]:
        return "Date de création prioritaire"

    else:
        return "Identifiant CRM"


df_decisions["critere_selection"] = (
    df_decisions.apply(
        determiner_critere_selection,
        axis=1
    )
)

# Afficher le rapport
print("JOURNAL DES DÉCISIONS DE SÉLECTION")
print("----------------------------------")

print("Nombre de décisions :", len(df_decisions))

print("\nRépartition des critères :")

display(
    df_decisions["critere_selection"].value_counts()
)

# Afficher quelques exemples
display(
    df_decisions[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite",
            "date_creation_fiable",
            "critere_selection"
        ]
    ].head(20)
)

JOURNAL DES DÉCISIONS DE SÉLECTION
----------------------------------
Nombre de décisions : 1693

Répartition des critères :


critere_selection
Identifiant CRM                 1543
Meilleur score de qualité        147
Date de création prioritaire       3
Name: count, dtype: int64

,crm_customer_id,prenom,nom,score_qualite,date_creation_fiable,critere_selection
0,CRM-0030630,Alexandre,Bélanger,100,2020-04-12,Meilleur score de qualité
1,CRM-0019248,Alexandre,Bergeron,100,2019-04-23,Identifiant CRM
2,CRM-0015890,Alexandre,Bergeron,100,2022-07-15,Identifiant CRM
3,CRM-0031465,Alexandre,Bergeron,100,2024-03-23,Identifiant CRM
4,CRM-0008675,ALEXANDRE,Bergeron,100,2023-02-07,Identifiant CRM
5,CRM-0007267,Alexandre,Bouchard,100,2024-06-25,Identifiant CRM
6,CRM-0005117,Alexandre,bouchard,100,2024-03-08,Identifiant CRM
7,CRM-0011785,ALEXANDRE,Caron,100,2019-01-02,Identifiant CRM
8,CRM-0020914,Alexandre,Caron,100,2022-07-11,Identifiant CRM
9,CRM-0018916,Alexandre,Côté,100,2019-07-13,Identifiant CRM


In [68]:
# Enregistrer le journal des décisions de sélection

from pathlib import Path

# Créer le dossier de documentation
Path("documentation").mkdir(exist_ok=True)

# Définir les colonnes à conserver
colonnes_journal = [
    "crm_customer_id",
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige",
    "score_qualite",
    "date_creation_fiable",
    "critere_selection"
]

# Préparer le journal
journal_decisions = df_decisions[
    colonnes_journal
].copy()

# Enregistrer le fichier CSV
chemin_journal = (
    "documentation/journal_decisions_selection_crm.csv"
)

journal_decisions.to_csv(
    chemin_journal,
    index=False,
    encoding="utf-8-sig"
)

# Vérifier l'enregistrement
print("JOURNAL DES DÉCISIONS CRM")
print("------------------------")

print("Fichier enregistré :", chemin_journal)

print(
    "Nombre de décisions enregistrées :",
    len(journal_decisions)
)

print(
    "Nombre de colonnes :",
    len(journal_decisions.columns)
)

# Afficher les premières lignes
display(journal_decisions.head(10))

JOURNAL DES DÉCISIONS CRM
------------------------
Fichier enregistré : documentation/journal_decisions_selection_crm.csv
Nombre de décisions enregistrées : 1693
Nombre de colonnes : 8


,crm_customer_id,prenom_normalise,nom_normalise,phone_normalise,email_corrige,score_qualite,date_creation_fiable,critere_selection
0,CRM-0030630,alexandre,belanger,4508447165,alexandre.belanger.44501@exemple.ca,100,2020-04-12,Meilleur score de qualité
1,CRM-0019248,alexandre,bergeron,4386958150,alexandre.bergeron.28153@exemple.ca,100,2019-04-23,Identifiant CRM
2,CRM-0015890,alexandre,bergeron,4389707005,alexandre.bergeron.23211@courriel.test,100,2022-07-15,Identifiant CRM
3,CRM-0031465,alexandre,bergeron,5145561938,alexandre.bergeron.45735@exemple.ca,100,2024-03-23,Identifiant CRM
4,CRM-0008675,alexandre,bergeron,8194312991,alexandre.bergeron.12713@exemple.ca,100,2023-02-07,Identifiant CRM
5,CRM-0007267,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,100,2024-06-25,Identifiant CRM
6,CRM-0005117,alexandre,bouchard,8194631539,alexandre.bouchard.7550@demo.invalid,100,2024-03-08,Identifiant CRM
7,CRM-0011785,alexandre,caron,4386834984,alexandre.caron.17212@courriel.test,100,2019-01-02,Identifiant CRM
8,CRM-0020914,alexandre,caron,5814936501,alexandre.caron.30535@exemple.ca,100,2022-07-11,Identifiant CRM
9,CRM-0018916,alexandre,cote,4505838134,alexandre.cote.27652@courriel.test,100,2019-07-13,Identifiant CRM


In [69]:
# Associer chaque doublon potentiel à sa fiche principale

# Définir les critères de rapprochement
criteres_rapprochement = [
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige"
]

# Récupérer les fiches principales
fiches_principales = df_selection.loc[
    df_selection["candidate_principale"],
    criteres_rapprochement + ["crm_customer_id"]
].copy()

# Renommer l'identifiant de la fiche principale
fiches_principales = fiches_principales.rename(
    columns={
        "crm_customer_id": "crm_id_principal"
    }
)

# Associer chaque fiche à son identifiant principal
df_correspondances = df_selection.merge(
    fiches_principales,
    on=criteres_rapprochement,
    how="left",
    validate="many_to_one"
)

# Identifier les fiches principales et secondaires
df_correspondances["type_fiche"] = (
    df_correspondances["candidate_principale"]
    .map({
        True: "Principale",
        False: "Secondaire"
    })
)

# Vérifier les résultats
print("PRÉPARATION DU GOLDEN RECORD")
print("----------------------------")

print(
    "Nombre total de fiches :",
    len(df_correspondances)
)

print(
    "Fiches principales :",
    (df_correspondances["type_fiche"] == "Principale").sum()
)

print(
    "Fiches secondaires :",
    (df_correspondances["type_fiche"] == "Secondaire").sum()
)

print(
    "Identifiants principaux manquants :",
    df_correspondances["crm_id_principal"].isna().sum()
)

# Afficher quelques correspondances
display(
    df_correspondances[
        [
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite",
            "crm_id_principal",
            "type_fiche"
        ]
    ].head(20)
)

PRÉPARATION DU GOLDEN RECORD
----------------------------
Nombre total de fiches : 3386
Fiches principales : 1693
Fiches secondaires : 1693
Identifiants principaux manquants : 0


,crm_customer_id,prenom,nom,score_qualite,crm_id_principal,type_fiche
0,CRM-0030630,Alexandre,Bélanger,100,CRM-0030630,Principale
1,CRM-0030629,Alexandre,Bélanger,80,CRM-0030630,Secondaire
2,CRM-0019248,Alexandre,Bergeron,100,CRM-0019248,Principale
3,CRM-0019249,Alexandre,Bergeron,100,CRM-0019248,Secondaire
4,CRM-0015890,Alexandre,Bergeron,100,CRM-0015890,Principale
5,CRM-0015891,ALEXANDRE,Bergeron,100,CRM-0015890,Secondaire
6,CRM-0031465,Alexandre,Bergeron,100,CRM-0031465,Principale
7,CRM-0031466,Alexandre,Bergeron,100,CRM-0031465,Secondaire
8,CRM-0008675,ALEXANDRE,Bergeron,100,CRM-0008675,Principale
9,CRM-0008676,Alexandre,Bergeron,100,CRM-0008675,Secondaire


In [70]:
# Vérification de la cohérence des correspondances

# Vérifier les fiches principales
principales = df_correspondances[
    df_correspondances["type_fiche"] == "Principale"
]

principales_valides = (
    principales["crm_customer_id"]
    == principales["crm_id_principal"]
)

# Vérifier les fiches secondaires
secondaires = df_correspondances[
    df_correspondances["type_fiche"] == "Secondaire"
]

secondaires_valides = (
    secondaires["crm_customer_id"]
    != secondaires["crm_id_principal"]
)

# Vérifier l'unicité de la fiche principale par groupe
verification_groupes = (
    df_correspondances
    .groupby(criteres_rapprochement)
    .agg(
        nombre_fiches=("crm_customer_id", "count"),
        nombre_identifiants_principaux=(
            "crm_id_principal", "nunique"
        )
    )
    .reset_index()
)

groupes_valides = (
    verification_groupes[
        "nombre_identifiants_principaux"
    ] == 1
)

# Afficher les résultats
print("AUDIT DE COHÉRENCE DES CORRESPONDANCES")
print("--------------------------------------")

print(
    "Fiches principales correctement associées :",
    principales_valides.sum(),
    "/",
    len(principales)
)

print(
    "Fiches secondaires correctement associées :",
    secondaires_valides.sum(),
    "/",
    len(secondaires)
)

print(
    "Groupes avec un identifiant principal unique :",
    groupes_valides.sum(),
    "/",
    len(verification_groupes)
)

print(
    "Groupes présentant une anomalie :",
    (~groupes_valides).sum()
)

# Afficher quelques groupes
display(verification_groupes.head(20))

AUDIT DE COHÉRENCE DES CORRESPONDANCES
--------------------------------------
Fiches principales correctement associées : 1693 / 1693
Fiches secondaires correctement associées : 1693 / 1693
Groupes avec un identifiant principal unique : 1693 / 1693
Groupes présentant une anomalie : 0


,prenom_normalise,nom_normalise,phone_normalise,email_corrige,nombre_fiches,nombre_identifiants_principaux
0,alexandre,belanger,4508447165,alexandre.belanger.44501@exemple.ca,2,1
1,alexandre,bergeron,4386958150,alexandre.bergeron.28153@exemple.ca,2,1
2,alexandre,bergeron,4389707005,alexandre.bergeron.23211@courriel.test,2,1
3,alexandre,bergeron,5145561938,alexandre.bergeron.45735@exemple.ca,2,1
4,alexandre,bergeron,8194312991,alexandre.bergeron.12713@exemple.ca,2,1
5,alexandre,bouchard,5145325447,alexandre.bouchard.10706@demo.invalid,2,1
6,alexandre,bouchard,8194631539,alexandre.bouchard.7550@demo.invalid,2,1
7,alexandre,caron,4386834984,alexandre.caron.17212@courriel.test,2,1
8,alexandre,caron,5814936501,alexandre.caron.30535@exemple.ca,2,1
9,alexandre,cote,4505838134,alexandre.cote.27652@courriel.test,2,1


In [71]:
# Analyse des informations complémentaires

import pandas as pd

# Définir les colonnes à analyser
colonnes_enrichissement = [
    "address",
    "city",
    "postal_code_normalise"
]

# Séparer les fiches principales et secondaires
df_principales = df_correspondances.loc[
    df_correspondances["type_fiche"] == "Principale"
].copy()

df_secondaires = df_correspondances.loc[
    df_correspondances["type_fiche"] == "Secondaire"
].copy()

# Récupérer les informations des fiches secondaires
informations_secondaires = df_secondaires[
    ["crm_id_principal"] + colonnes_enrichissement
].copy()

# Renommer les colonnes secondaires
informations_secondaires = informations_secondaires.rename(
    columns={
        colonne: colonne + "_secondaire"
        for colonne in colonnes_enrichissement
    }
)

# Associer les informations secondaires aux fiches principales
df_enrichissement = df_principales.merge(
    informations_secondaires,
    on="crm_id_principal",
    how="left",
    validate="one_to_one"
)

# Identifier les valeurs manquantes ou vides
def valeur_manquante(serie):
    return (
        serie.isna()
        | serie.astype("string").str.strip().eq("").fillna(False)
    )

# Identifier les possibilités d'enrichissement
for colonne in colonnes_enrichissement:

    df_enrichissement[
        "enrichissement_" + colonne
    ] = (
        valeur_manquante(df_enrichissement[colonne])
        & ~valeur_manquante(
            df_enrichissement[colonne + "_secondaire"]
        )
    )

# Afficher les résultats
print("ANALYSE DES POSSIBILITÉS D'ENRICHISSEMENT")
print("-----------------------------------------")

print(
    "Fiches principales analysées :",
    len(df_enrichissement)
)

for colonne in colonnes_enrichissement:

    print(
        "Enrichissements possibles pour",
        colonne,
        ":",
        df_enrichissement[
            "enrichissement_" + colonne
        ].sum()
    )

# Afficher quelques exemples
display(
    df_enrichissement[
        [
            "crm_id_principal",
            "address",
            "address_secondaire",
            "city",
            "city_secondaire",
            "postal_code_normalise",
            "postal_code_normalise_secondaire"
        ]
    ].head(20)
)

ANALYSE DES POSSIBILITÉS D'ENRICHISSEMENT
-----------------------------------------
Fiches principales analysées : 1693
Enrichissements possibles pour address : 0
Enrichissements possibles pour city : 0
Enrichissements possibles pour postal_code_normalise : 3


,crm_id_principal,address,address_secondaire,city,city_secondaire,postal_code_normalise,postal_code_normalise_secondaire
0,CRM-0030630,2715 rue du Parc,2715 rue du Parc,Sherbrooke,Sherbrooke,L2Z2G1,L2Z2G1
1,CRM-0019248,2758 rue de la Gare,2758 rue de la Gare,Montréal,Montréal,J0L1J5,J0L1J5
2,CRM-0015890,2143 rue des Pins,2143 rue des Pins,Laval,Laval,J3L6L9,J3L6L9
3,CRM-0031465,8456 rue de l’Église,8456 rue de l’Église,Drummondville,Drummondville,J1V9R3,J1V9R3
4,CRM-0008675,9800 rue des Érables,9800 rue des Érables,Longueuil,Longueuil,H8G3X3,H8G3X3
5,CRM-0007267,5322 rue des Érables,5322 rue des Érables,Montréal,Montreal,J7J8L1,J7J8L1
6,CRM-0005117,3259 rue des Érables,3259 rue des Érables,Québec,Québec,G5Z4X5,G5Z4X5
7,CRM-0011785,8095 rue du Fleuve,8095 rue du Fleuve,Drummondville,Drummondville,K4Z0C3,K4Z0C3
8,CRM-0020914,5005 rue de l’Église,5005 rue de l’Église,Trois-Rivières,Trois-Rivières,H7K7R7,H7K7R7
9,CRM-0018916,906 rue de la Gare,906 rue de la Gare,Saguenay,Saguenay,H8R8T4,H8R8T4


In [72]:
# Vérification des codes postaux récupérables

# Sélectionner les fiches pouvant être enrichies
df_codes_recuperables = df_enrichissement.loc[
    df_enrichissement["enrichissement_postal_code_normalise"]
].copy()

# Afficher les résultats
print("AUDIT DES CODES POSTAUX RÉCUPÉRABLES")
print("-----------------------------------")

print(
    "Nombre de fiches concernées :",
    len(df_codes_recuperables)
)

# Afficher les informations géographiques
colonnes_verification = [
    "crm_id_principal",
    "prenom",
    "nom",
    "address",
    "address_secondaire",
    "city",
    "city_secondaire",
    "postal_code_normalise",
    "postal_code_normalise_secondaire"
]

display(
    df_codes_recuperables[
        colonnes_verification
    ]
)

# Vérifier le format des codes postaux secondaires
regex_postal = r"^[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z]\d[ABCEGHJ-NPRSTV-Z]\d$"

df_codes_recuperables["code_postal_secondaire_valide"] = (
    df_codes_recuperables[
        "postal_code_normalise_secondaire"
    ]
    .astype("string")
    .str.fullmatch(regex_postal)
    .fillna(False)
)

print("\nVALIDATION DES CODES POSTAUX")
print("----------------------------")

print(
    "Codes postaux au format valide :",
    df_codes_recuperables[
        "code_postal_secondaire_valide"
    ].sum()
)

print(
    "Codes postaux au format invalide :",
    (~df_codes_recuperables[
        "code_postal_secondaire_valide"
    ]).sum()
)

display(
    df_codes_recuperables[
        [
            "crm_id_principal",
            "city",
            "postal_code_normalise_secondaire",
            "code_postal_secondaire_valide"
        ]
    ]
)

AUDIT DES CODES POSTAUX RÉCUPÉRABLES
-----------------------------------
Nombre de fiches concernées : 3


,crm_id_principal,prenom,nom,address,address_secondaire,city,city_secondaire,postal_code_normalise,postal_code_normalise_secondaire
987,CRM-0032609,LOUIS,Tremblay,4463 rue Saint-Jean,4463 rue Saint-Jean,Québec,Québec,<NA>,G8Z0R9
1304,CRM-0033379,Nathalie,Gauthier,728 rue du Moulin,728 rue du Moulin,Saguenay,Saguenay,<NA>,G6E6E3
1654,CRM-0012578,Sophie,Dubois,4793 rue des Pins,4793 rue des Pins,Longueuil,Longueuil,<NA>,K1R1P9



VALIDATION DES CODES POSTAUX
----------------------------
Codes postaux au format valide : 3
Codes postaux au format invalide : 0


,crm_id_principal,city,postal_code_normalise_secondaire,code_postal_secondaire_valide
987,CRM-0032609,Québec,G8Z0R9,True
1304,CRM-0033379,Saguenay,G6E6E3,True
1654,CRM-0012578,Longueuil,K1R1P9,True


In [73]:
# Contrôle de cohérence géographique des codes postaux

# Définir les premières lettres attendues
prefixes_villes = {
    "Québec": "G",
    "Saguenay": "G",
    "Longueuil": "J"
}

# Récupérer le préfixe du code postal
df_codes_recuperables["prefixe_postal"] = (
    df_codes_recuperables[
        "postal_code_normalise_secondaire"
    ]
    .astype("string")
    .str[0]
)

# Définir le préfixe attendu selon la ville
df_codes_recuperables["prefixe_attendu"] = (
    df_codes_recuperables["city"]
    .map(prefixes_villes)
)

# Vérifier la cohérence géographique simplifiée
df_codes_recuperables["coherence_geographique"] = (
    df_codes_recuperables["prefixe_postal"]
    == df_codes_recuperables["prefixe_attendu"]
)

# Définir le statut de validation
df_codes_recuperables["statut_enrichissement"] = (
    df_codes_recuperables["coherence_geographique"]
    .map({
        True: "À vérifier avec l'adresse",
        False: "Contrôle manuel nécessaire"
    })
)

# Afficher les résultats
print("AUDIT DE COHÉRENCE GÉOGRAPHIQUE")
print("--------------------------------")

print(
    "Codes postaux analysés :",
    len(df_codes_recuperables)
)

print(
    "Préfixes cohérents avec la ville :",
    df_codes_recuperables[
        "coherence_geographique"
    ].sum()
)

print(
    "Préfixes incohérents avec la ville :",
    (~df_codes_recuperables[
        "coherence_geographique"
    ]).sum()
)

# Afficher les détails
display(
    df_codes_recuperables[
        [
            "crm_id_principal",
            "city",
            "postal_code_normalise_secondaire",
            "prefixe_postal",
            "prefixe_attendu",
            "coherence_geographique",
            "statut_enrichissement"
        ]
    ]
)

AUDIT DE COHÉRENCE GÉOGRAPHIQUE
--------------------------------
Codes postaux analysés : 3
Préfixes cohérents avec la ville : 2
Préfixes incohérents avec la ville : 1


,crm_id_principal,city,postal_code_normalise_secondaire,prefixe_postal,prefixe_attendu,coherence_geographique,statut_enrichissement
987,CRM-0032609,Québec,G8Z0R9,G,G,True,À vérifier avec l'adresse
1304,CRM-0033379,Saguenay,G6E6E3,G,G,True,À vérifier avec l'adresse
1654,CRM-0012578,Longueuil,K1R1P9,K,J,False,Contrôle manuel nécessaire


In [74]:
# Sécurisation des décisions d'enrichissement

from pathlib import Path

# Créer une copie de travail
df_validation_enrichissement = df_codes_recuperables.copy()

# Définir les décisions selon les contrôles réalisés
df_validation_enrichissement["decision_enrichissement"] = (
    df_validation_enrichissement["coherence_geographique"]
    .map({
        True: "En attente de validation géographique",
        False: "Rejet automatique - préfixe incohérent"
    })
)

# Aucune modification automatique sans validation complète
df_validation_enrichissement["enrichissement_autorise"] = False

# Préparer le journal de validation
colonnes_journal = [
    "crm_id_principal",
    "city",
    "postal_code_normalise_secondaire",
    "code_postal_secondaire_valide",
    "coherence_geographique",
    "decision_enrichissement",
    "enrichissement_autorise"
]

journal_enrichissement = (
    df_validation_enrichissement[colonnes_journal]
    .copy()
)

# Créer le dossier de documentation
Path("documentation").mkdir(exist_ok=True)

# Enregistrer le journal
chemin_journal = (
    "documentation/journal_validation_enrichissement.csv"
)

journal_enrichissement.to_csv(
    chemin_journal,
    index=False,
    encoding="utf-8-sig"
)

# Afficher les résultats
print("JOURNAL DE VALIDATION DES ENRICHISSEMENTS")
print("----------------------------------------")

print(
    "Fiches examinées :",
    len(journal_enrichissement)
)

print(
    "Enrichissements autorisés :",
    journal_enrichissement[
        "enrichissement_autorise"
    ].sum()
)

print(
    "Fiches nécessitant une validation :",
    (
        journal_enrichissement["decision_enrichissement"]
        == "En attente de validation géographique"
    ).sum()
)

print(
    "Fiches rejetées automatiquement :",
    (
        journal_enrichissement["decision_enrichissement"]
        == "Rejet automatique - préfixe incohérent"
    ).sum()
)

print("\nFichier enregistré :", chemin_journal)

display(journal_enrichissement)

JOURNAL DE VALIDATION DES ENRICHISSEMENTS
----------------------------------------
Fiches examinées : 3
Enrichissements autorisés : 0
Fiches nécessitant une validation : 2
Fiches rejetées automatiquement : 1

Fichier enregistré : documentation/journal_validation_enrichissement.csv


,crm_id_principal,city,postal_code_normalise_secondaire,code_postal_secondaire_valide,coherence_geographique,decision_enrichissement,enrichissement_autorise
987,CRM-0032609,Québec,G8Z0R9,True,True,En attente de validation géographique,False
1304,CRM-0033379,Saguenay,G6E6E3,True,True,En attente de validation géographique,False
1654,CRM-0012578,Longueuil,K1R1P9,True,False,Rejet automatique - préfixe incohérent,False


In [75]:
# Création des Golden Records
# Construction des fiches principales

# Récupérer les fiches principales
df_golden_records = df_correspondances.loc[
    df_correspondances["type_fiche"] == "Principale"
].copy()

# Ajouter le statut de consolidation
df_golden_records["statut_consolidation"] = (
    "Golden Record provisoire"
)

# Conserver l'identifiant principal
df_golden_records["golden_record_id"] = (
    df_golden_records["crm_id_principal"]
)

# Vérifier les résultats
print("CRÉATION DES GOLDEN RECORDS")
print("---------------------------")

print(
    "Nombre de Golden Records :",
    len(df_golden_records)
)

print(
    "Identifiants uniques :",
    df_golden_records["golden_record_id"].nunique()
)

print(
    "Identifiants manquants :",
    df_golden_records["golden_record_id"].isna().sum()
)

print(
    "Doublons d'identifiants :",
    df_golden_records["golden_record_id"].duplicated().sum()
)

# Afficher les premières fiches
display(
    df_golden_records[
        [
            "golden_record_id",
            "crm_customer_id",
            "prenom",
            "nom",
            "score_qualite",
            "statut_consolidation"
        ]
    ].head(20)
)

CRÉATION DES GOLDEN RECORDS
---------------------------
Nombre de Golden Records : 1693
Identifiants uniques : 1693
Identifiants manquants : 0
Doublons d'identifiants : 0


,golden_record_id,crm_customer_id,prenom,nom,score_qualite,statut_consolidation
0,CRM-0030630,CRM-0030630,Alexandre,Bélanger,100,Golden Record provisoire
2,CRM-0019248,CRM-0019248,Alexandre,Bergeron,100,Golden Record provisoire
4,CRM-0015890,CRM-0015890,Alexandre,Bergeron,100,Golden Record provisoire
6,CRM-0031465,CRM-0031465,Alexandre,Bergeron,100,Golden Record provisoire
8,CRM-0008675,CRM-0008675,ALEXANDRE,Bergeron,100,Golden Record provisoire
10,CRM-0007267,CRM-0007267,Alexandre,Bouchard,100,Golden Record provisoire
12,CRM-0005117,CRM-0005117,Alexandre,bouchard,100,Golden Record provisoire
14,CRM-0011785,CRM-0011785,ALEXANDRE,Caron,100,Golden Record provisoire
16,CRM-0020914,CRM-0020914,Alexandre,Caron,100,Golden Record provisoire
18,CRM-0018916,CRM-0018916,Alexandre,Côté,100,Golden Record provisoire


In [76]:
# Vérification de la qualité des Golden Records

# 1. Vérifier les scores de qualité

print("AUDIT DE QUALITÉ DES GOLDEN RECORDS")
print("-----------------------------------")

print(
    "Nombre de Golden Records :",
    len(df_golden_records)
)

print(
    "Score moyen :",
    round(df_golden_records["score_qualite"].mean(), 2)
)

print(
    "Score minimum :",
    df_golden_records["score_qualite"].min()
)

print(
    "Score maximum :",
    df_golden_records["score_qualite"].max()
)

# 2. Vérifier les dates fiables

print(
    "\nDates de création fiables disponibles :",
    df_golden_records["date_creation_fiable"].notna().sum()
)

print(
    "Dates de création fiables manquantes :",
    df_golden_records["date_creation_fiable"].isna().sum()
)

# 3. Vérifier les informations essentielles

colonnes_essentielles = [
    "crm_customer_id",
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige"
]

print("\nVALEURS MANQUANTES PAR COLONNE")
print("-----------------------------")

display(
    df_golden_records[
        colonnes_essentielles
    ].isna().sum()
)

# 4. Identifier les fiches nécessitant une vérification

df_golden_records["verification_requise"] = (
    (df_golden_records["score_qualite"] < 70)
    | df_golden_records["date_creation_fiable"].isna()
    | df_golden_records[colonnes_essentielles].isna().any(axis=1)
)

# 5. Afficher les résultats

print(
    "\nGolden Records nécessitant une vérification :",
    df_golden_records["verification_requise"].sum()
)

print(
    "Golden Records sans anomalie selon ces contrôles :",
    (~df_golden_records["verification_requise"]).sum()
)

# 6. Afficher quelques exemples

display(
    df_golden_records[
        [
            "golden_record_id",
            "prenom",
            "nom",
            "score_qualite",
            "date_creation_fiable",
            "verification_requise"
        ]
    ].head(20)
)

AUDIT DE QUALITÉ DES GOLDEN RECORDS
-----------------------------------
Nombre de Golden Records : 1693
Score moyen : 99.95
Score minimum : 80
Score maximum : 100

Dates de création fiables disponibles : 1692
Dates de création fiables manquantes : 1

VALEURS MANQUANTES PAR COLONNE
-----------------------------


crm_customer_id     0
prenom_normalise    0
nom_normalise       0
phone_normalise     0
email_corrige       0
dtype: int64


Golden Records nécessitant une vérification : 1
Golden Records sans anomalie selon ces contrôles : 1692


,golden_record_id,prenom,nom,score_qualite,date_creation_fiable,verification_requise
0,CRM-0030630,Alexandre,Bélanger,100,2020-04-12,False
2,CRM-0019248,Alexandre,Bergeron,100,2019-04-23,False
4,CRM-0015890,Alexandre,Bergeron,100,2022-07-15,False
6,CRM-0031465,Alexandre,Bergeron,100,2024-03-23,False
8,CRM-0008675,ALEXANDRE,Bergeron,100,2023-02-07,False
10,CRM-0007267,Alexandre,Bouchard,100,2024-06-25,False
12,CRM-0005117,Alexandre,bouchard,100,2024-03-08,False
14,CRM-0011785,ALEXANDRE,Caron,100,2019-01-02,False
16,CRM-0020914,Alexandre,Caron,100,2022-07-11,False
18,CRM-0018916,Alexandre,Côté,100,2019-07-13,False


In [78]:
# 3. Examiner les informations de la fiche

colonnes_verification = [
    "golden_record_id",
    "crm_customer_id",
    "prenom",
    "nom",
    "score_qualite",
    "date_creation",
    "date_creation_fiable",
    "verification_requise"
]

display(
    df_golden_a_verifier[
        colonnes_verification
    ]
)

,golden_record_id,crm_customer_id,prenom,nom,score_qualite,date_creation,date_creation_fiable,verification_requise
2614,CRM-0005049,CRM-0005049,Nathalie,girard,80,2032-01-01,NaT,True


In [79]:
# Documenter le Golden Record à vérifier

# Créer une copie de travail
df_golden_a_verifier = df_golden_a_verifier.copy()

# Identifier le motif de vérification
df_golden_a_verifier["motif_verification"] = (
    "Date de création à examiner"
)

df_golden_a_verifier.loc[
    df_golden_a_verifier["date_creation_fiable"].isna(),
    "motif_verification"
] = "Date de création fiable manquante"

# Ajouter le statut de validation
df_golden_a_verifier["statut_validation"] = (
    "En attente de vérification"
)

# Afficher les résultats
print("RÉSULTAT DE LA VÉRIFICATION")
print("---------------------------")

display(
    df_golden_a_verifier[
        [
            "golden_record_id",
            "date_creation",
            "date_creation_fiable",
            "motif_verification",
            "statut_validation"
        ]
    ]
)

RÉSULTAT DE LA VÉRIFICATION
---------------------------


,golden_record_id,date_creation,date_creation_fiable,motif_verification,statut_validation
2614,CRM-0005049,2032-01-01,NaT,Date de création fiable manquante,En attente de vérification


In [80]:
# Enregistrement du journal des anomalies

from pathlib import Path

# Créer le dossier de documentation
Path("documentation").mkdir(exist_ok=True)

# Définir les colonnes à conserver
colonnes_anomalies = [
    "golden_record_id",
    "crm_customer_id",
    "prenom",
    "nom",
    "date_creation",
    "date_creation_fiable",
    "score_qualite",
    "motif_verification",
    "statut_validation"
]

# Préparer le journal
journal_anomalies = df_golden_a_verifier[
    colonnes_anomalies
].copy()

# Définir le chemin du fichier
chemin_anomalies = (
    "documentation/journal_anomalies_golden_records.csv"
)

# Enregistrer au format CSV
journal_anomalies.to_csv(
    chemin_anomalies,
    index=False,
    encoding="utf-8-sig"
)

# Vérifier les résultats
print("JOURNAL DES ANOMALIES DES GOLDEN RECORDS")
print("---------------------------------------")

print("Fichier enregistré :", chemin_anomalies)

print(
    "Nombre d'anomalies documentées :",
    len(journal_anomalies)
)

print(
    "Nombre de colonnes :",
    len(journal_anomalies.columns)
)

# Afficher le journal
display(journal_anomalies)

JOURNAL DES ANOMALIES DES GOLDEN RECORDS
---------------------------------------
Fichier enregistré : documentation/journal_anomalies_golden_records.csv
Nombre d'anomalies documentées : 1
Nombre de colonnes : 9


,golden_record_id,crm_customer_id,prenom,nom,date_creation,date_creation_fiable,score_qualite,motif_verification,statut_validation
2614,CRM-0005049,CRM-0005049,Nathalie,girard,2032-01-01,NaT,80,Date de création fiable manquante,En attente de vérification


In [81]:
# Audit final des Golden Records

print("AUDIT FINAL DES GOLDEN RECORDS")
print("--------------------------------")

# 1. Nombre total de Golden Records
total_golden = len(df_golden_records)

print("Nombre total de Golden Records :", total_golden)

# 2. Vérifier l'unicité des identifiants
identifiants_uniques = df_golden_records[
    "golden_record_id"
].nunique()

doublons_identifiants = df_golden_records[
    "golden_record_id"
].duplicated().sum()

print("Identifiants uniques :", identifiants_uniques)
print("Identifiants dupliqués :", doublons_identifiants)

# 3. Vérifier les valeurs manquantes
colonnes_essentielles = [
    "golden_record_id",
    "crm_customer_id",
    "prenom_normalise",
    "nom_normalise",
    "phone_normalise",
    "email_corrige"
]

print("\nVALEURS MANQUANTES")
print("-------------------")

display(
    df_golden_records[
        colonnes_essentielles
    ].isna().sum()
)

# 4. Vérifier les scores de qualité
print("\nSCORES DE QUALITÉ")
print("-------------------")

print(
    "Score moyen :",
    round(df_golden_records["score_qualite"].mean(), 2)
)

print(
    "Score minimum :",
    df_golden_records["score_qualite"].min()
)

print(
    "Score maximum :",
    df_golden_records["score_qualite"].max()
)

# 5. Vérifier les anomalies
nombre_anomalies = df_golden_records[
    "verification_requise"
].sum()

print("\nCONTRÔLE DES ANOMALIES")
print("----------------------")

print("Golden Records à vérifier :", nombre_anomalies)

print(
    "Golden Records sans anomalie détectée :",
    total_golden - nombre_anomalies
)

# 6. Vérifier la cohérence globale
audit_valide = (
    total_golden == 1693
    and identifiants_uniques == total_golden
    and doublons_identifiants == 0
    and df_golden_records[colonnes_essentielles].notna().all().all()
    and df_golden_records["score_qualite"].between(0, 100).all()
    and nombre_anomalies == len(journal_anomalies)
)

print("\nRÉSULTAT GLOBAL DE L'AUDIT")
print("--------------------------")

if audit_valide:
    print("AUDIT VALIDÉ : cohérence structurelle confirmée.")
else:
    print("AUDIT NON VALIDÉ : des contrôles nécessitent une correction.")

AUDIT FINAL DES GOLDEN RECORDS
--------------------------------
Nombre total de Golden Records : 1693
Identifiants uniques : 1693
Identifiants dupliqués : 0

VALEURS MANQUANTES
-------------------


golden_record_id    0
crm_customer_id     0
prenom_normalise    0
nom_normalise       0
phone_normalise     0
email_corrige       0
dtype: int64


SCORES DE QUALITÉ
-------------------
Score moyen : 99.95
Score minimum : 80
Score maximum : 100

CONTRÔLE DES ANOMALIES
----------------------
Golden Records à vérifier : 1
Golden Records sans anomalie détectée : 1692

RÉSULTAT GLOBAL DE L'AUDIT
--------------------------
AUDIT VALIDÉ : cohérence structurelle confirmée.


In [82]:
# Exportation des Golden Records consolidés

from pathlib import Path
import pandas as pd

# 1. Créer le dossier d'exportation
dossier_export = Path("exports")
dossier_export.mkdir(exist_ok=True)

# 2. Préparer une copie des Golden Records
df_export_golden = df_golden_records.copy()

# 3. Ajouter le statut de validation
df_export_golden["statut_validation"] = (
    "Sans anomalie détectée"
)

df_export_golden.loc[
    df_export_golden["verification_requise"],
    "statut_validation"
] = "En attente de vérification"

# 4. Définir le chemin du fichier
chemin_export = (
    dossier_export / "golden_records_consolides.csv"
)

# 5. Exporter les données
df_export_golden.to_csv(
    chemin_export,
    index=False,
    encoding="utf-8-sig"
)

# 6. Vérifier le fichier exporté
df_controle_export = pd.read_csv(
    chemin_export,
    dtype={"golden_record_id": "string",
           "crm_customer_id": "string"}
)

print("EXPORTATION DES GOLDEN RECORDS")
print("--------------------------------")

print("Fichier enregistré :", chemin_export)

print(
    "Nombre de Golden Records exportés :",
    len(df_controle_export)
)

print(
    "Nombre de colonnes :",
    len(df_controle_export.columns)
)

print(
    "Identifiants uniques :",
    df_controle_export["golden_record_id"].nunique()
)

print(
    "Identifiants dupliqués :",
    df_controle_export["golden_record_id"].duplicated().sum()
)

print("\nRÉPARTITION DES STATUTS")
print("------------------------")

display(
    df_controle_export["statut_validation"].value_counts()
)

# 7. Vérifier la cohérence de l'exportation
export_valide = (
    len(df_controle_export) == len(df_golden_records)
    and df_controle_export["golden_record_id"].is_unique
    and df_controle_export["golden_record_id"].notna().all()
    and (
        df_controle_export["statut_validation"]
        == "En attente de vérification"
    ).sum() == nombre_anomalies
)

print("\nRÉSULTAT DU CONTRÔLE")
print("---------------------")

if export_valide:
    print("EXPORT VALIDÉ : cohérence confirmée.")
else:
    print("EXPORT NON VALIDÉ : vérification nécessaire.")

# 8. Afficher quelques Golden Records
display(
    df_controle_export[
        [
            "golden_record_id",
            "prenom",
            "nom",
            "score_qualite",
            "statut_validation"
        ]
    ].head(10)
)

EXPORTATION DES GOLDEN RECORDS
--------------------------------
Fichier enregistré : exports\golden_records_consolides.csv
Nombre de Golden Records exportés : 1693
Nombre de colonnes : 33
Identifiants uniques : 1693
Identifiants dupliqués : 0

RÉPARTITION DES STATUTS
------------------------


statut_validation
Sans anomalie détectée        1692
En attente de vérification       1
Name: count, dtype: int64


RÉSULTAT DU CONTRÔLE
---------------------
EXPORT VALIDÉ : cohérence confirmée.


,golden_record_id,prenom,nom,score_qualite,statut_validation
0,CRM-0030630,Alexandre,Bélanger,100,Sans anomalie détectée
1,CRM-0019248,Alexandre,Bergeron,100,Sans anomalie détectée
2,CRM-0015890,Alexandre,Bergeron,100,Sans anomalie détectée
3,CRM-0031465,Alexandre,Bergeron,100,Sans anomalie détectée
4,CRM-0008675,ALEXANDRE,Bergeron,100,Sans anomalie détectée
5,CRM-0007267,Alexandre,Bouchard,100,Sans anomalie détectée
6,CRM-0005117,Alexandre,bouchard,100,Sans anomalie détectée
7,CRM-0011785,ALEXANDRE,Caron,100,Sans anomalie détectée
8,CRM-0020914,Alexandre,Caron,100,Sans anomalie détectée
9,CRM-0018916,Alexandre,Côté,100,Sans anomalie détectée


In [83]:
# Exportation de la table de correspondance CRM

from pathlib import Path
import pandas as pd

# 1. Créer le dossier d'exportation
dossier_export = Path("exports")
dossier_export.mkdir(exist_ok=True)

# 2. Définir les colonnes de correspondance
colonnes_correspondance = [
    "crm_customer_id",
    "crm_id_principal",
    "type_fiche",
    "score_qualite"
]

# 3. Préparer la table
df_mapping_crm = df_correspondances[
    colonnes_correspondance
].copy()

# 4. Renommer l'identifiant principal
df_mapping_crm = df_mapping_crm.rename(
    columns={
        "crm_id_principal": "golden_record_id"
    }
)

# 5. Définir le chemin d'exportation
chemin_mapping = (
    dossier_export / "correspondance_crm_golden_records.csv"
)

# 6. Exporter le fichier CSV
df_mapping_crm.to_csv(
    chemin_mapping,
    index=False,
    encoding="utf-8-sig"
)

# 7. Vérifier les résultats
print("EXPORTATION DES CORRESPONDANCES CRM")
print("----------------------------------")

print("Fichier enregistré :", chemin_mapping)

print(
    "Nombre de correspondances :",
    len(df_mapping_crm)
)

print(
    "Golden Records référencés :",
    df_mapping_crm["golden_record_id"].nunique()
)

print(
    "Identifiants CRM dupliqués :",
    df_mapping_crm["crm_customer_id"].duplicated().sum()
)

print(
    "Identifiants principaux manquants :",
    df_mapping_crm["golden_record_id"].isna().sum()
)

print("\nRÉPARTITION DES TYPES DE FICHES")
print("--------------------------------")

display(
    df_mapping_crm["type_fiche"].value_counts()
)

# 8. Vérifier l'intégrité des correspondances
ids_golden = set(df_golden_records["golden_record_id"])

correspondances_valides = (
    len(df_mapping_crm) == len(df_correspondances)
    and df_mapping_crm["crm_customer_id"].is_unique
    and df_mapping_crm["golden_record_id"].notna().all()
    and df_mapping_crm["golden_record_id"].isin(ids_golden).all()
)

print("\nRÉSULTAT DU CONTRÔLE")
print("--------------------")

if correspondances_valides:
    print("CORRESPONDANCES VALIDÉES.")
else:
    print("ANOMALIES DÉTECTÉES : vérification nécessaire.")

# 9. Afficher les premières correspondances
display(df_mapping_crm.head(20))

EXPORTATION DES CORRESPONDANCES CRM
----------------------------------
Fichier enregistré : exports\correspondance_crm_golden_records.csv
Nombre de correspondances : 3386
Golden Records référencés : 1693
Identifiants CRM dupliqués : 0
Identifiants principaux manquants : 0

RÉPARTITION DES TYPES DE FICHES
--------------------------------


type_fiche
Principale    1693
Secondaire    1693
Name: count, dtype: int64


RÉSULTAT DU CONTRÔLE
--------------------
CORRESPONDANCES VALIDÉES.


,crm_customer_id,golden_record_id,type_fiche,score_qualite
0,CRM-0030630,CRM-0030630,Principale,100
1,CRM-0030629,CRM-0030630,Secondaire,80
2,CRM-0019248,CRM-0019248,Principale,100
3,CRM-0019249,CRM-0019248,Secondaire,100
4,CRM-0015890,CRM-0015890,Principale,100
5,CRM-0015891,CRM-0015890,Secondaire,100
6,CRM-0031465,CRM-0031465,Principale,100
7,CRM-0031466,CRM-0031465,Secondaire,100
8,CRM-0008675,CRM-0008675,Principale,100
9,CRM-0008676,CRM-0008675,Secondaire,100


In [84]:
# Contrôle d'intégrité des fichiers exportés

import pandas as pd

# 1. Charger les fichiers exportés
df_golden_export = pd.read_csv(
    "exports/golden_records_consolides.csv",
    dtype={
        "golden_record_id": "string",
        "crm_customer_id": "string"
    }
)

df_mapping_export = pd.read_csv(
    "exports/correspondance_crm_golden_records.csv",
    dtype={
        "crm_customer_id": "string",
        "golden_record_id": "string"
    }
)

# 2. Vérifier les références vers les Golden Records
ids_golden = set(df_golden_export["golden_record_id"])

references_invalides = (
    ~df_mapping_export["golden_record_id"].isin(ids_golden)
).sum()

# 3. Vérifier les fiches principales
df_principales = df_mapping_export.loc[
    df_mapping_export["type_fiche"] == "Principale"
].copy()

# Vérifier que chaque fiche principale possède son propre identifiant
principales_incorrectes = (
    df_principales["crm_customer_id"]
    != df_principales["golden_record_id"]
).sum()

# Compter les fiches principales par Golden Record
nombre_principales = (
    df_principales
    .groupby("golden_record_id")
    .size()
    .reindex(df_golden_export["golden_record_id"], fill_value=0)
)

groupes_principaux_invalides = (
    nombre_principales != 1
).sum()

# 4. Vérifier les fiches secondaires
df_secondaires = df_mapping_export.loc[
    df_mapping_export["type_fiche"] == "Secondaire"
].copy()

secondaires_invalides = (
    df_secondaires["crm_customer_id"]
    == df_secondaires["golden_record_id"]
).sum()

# 5. Vérifier les identifiants CRM dupliqués
doublons_crm = (
    df_mapping_export["crm_customer_id"]
    .duplicated()
    .sum()
)

# 6. Afficher les résultats
print("AUDIT D'INTÉGRITÉ DES EXPORTATIONS")
print("---------------------------------")

print("Golden Records :", len(df_golden_export))
print("Correspondances CRM :", len(df_mapping_export))

print("Références invalides :", references_invalides)

print(
    "Fiches principales incorrectes :",
    principales_incorrectes
)

print(
    "Golden Records sans une unique fiche principale :",
    groupes_principaux_invalides
)

print(
    "Fiches secondaires incorrectes :",
    secondaires_invalides
)

print("Identifiants CRM dupliqués :", doublons_crm)

# 7. Vérifier l'intégrité globale
integrite_valide = (
    len(df_golden_export) == 1693
    and len(df_mapping_export) == 3386
    and df_golden_export["golden_record_id"].is_unique
    and df_mapping_export["crm_customer_id"].is_unique
    and df_golden_export["golden_record_id"].notna().all()
    and df_mapping_export[
        ["crm_customer_id", "golden_record_id", "type_fiche"]
    ].notna().all().all()
    and df_mapping_export["type_fiche"].isin(
        ["Principale", "Secondaire"]
    ).all()
    and references_invalides == 0
    and principales_incorrectes == 0
    and groupes_principaux_invalides == 0
    and secondaires_invalides == 0
    and doublons_crm == 0
)

print("\nRÉSULTAT FINAL")
print("--------------")

if integrite_valide:
    print("INTÉGRITÉ VALIDÉE : les fichiers sont cohérents.")
else:
    print("INTÉGRITÉ NON VALIDÉE : corrections nécessaires.")

AUDIT D'INTÉGRITÉ DES EXPORTATIONS
---------------------------------
Golden Records : 1693
Correspondances CRM : 3386
Références invalides : 0
Fiches principales incorrectes : 0
Golden Records sans une unique fiche principale : 0
Fiches secondaires incorrectes : 0
Identifiants CRM dupliqués : 0

RÉSULTAT FINAL
--------------
INTÉGRITÉ VALIDÉE : les fichiers sont cohérents.


In [85]:
# Bilan final du dédoublonnage CRM

import pandas as pd

# 1. Récupérer les indicateurs principaux
total_fiches = len(df_mapping_export)

total_golden = len(df_golden_export)

fiches_principales = (
    df_mapping_export["type_fiche"] == "Principale"
).sum()

fiches_secondaires = (
    df_mapping_export["type_fiche"] == "Secondaire"
).sum()

fiches_a_verifier = (
    df_golden_export["statut_validation"]
    == "En attente de vérification"
).sum()

fiches_sans_anomalie = (
    df_golden_export["statut_validation"]
    == "Sans anomalie détectée"
).sum()

# 2. Calculer les indicateurs
taux_reduction = (
    fiches_secondaires / total_fiches * 100
    if total_fiches > 0 else 0
)

taux_qualite = (
    fiches_sans_anomalie / total_golden * 100
    if total_golden > 0 else 0
)

# 3. Construire le tableau récapitulatif
bilan_final = pd.DataFrame({
    "Indicateur": [
        "Fiches CRM analysées",
        "Golden Records créés",
        "Fiches principales",
        "Fiches secondaires",
        "Réduction des fiches redondantes (%)",
        "Golden Records sans anomalie détectée",
        "Golden Records à vérifier",
        "Taux sans anomalie détectée (%)",
        "Références CRM invalides",
        "Identifiants CRM dupliqués"
    ],
    "Valeur": [
        total_fiches,
        total_golden,
        fiches_principales,
        fiches_secondaires,
        round(taux_reduction, 2),
        fiches_sans_anomalie,
        fiches_a_verifier,
        round(taux_qualite, 2),
        references_invalides,
        doublons_crm
    ]
})

# 4. Afficher le bilan
print("BILAN FINAL DU DÉDOUBLONNAGE CRM")
print("--------------------------------")

display(bilan_final)

# 5. Afficher la conclusion
print("\nSYNTHÈSE DU PROJET")
print("------------------")

print("Fiches CRM analysées :", total_fiches)
print("Golden Records créés :", total_golden)
print("Fiches secondaires identifiées :", fiches_secondaires)

print(
    "Réduction des fiches redondantes :",
    round(taux_reduction, 2), "%"
)

print(
    "Golden Records sans anomalie détectée :",
    fiches_sans_anomalie
)

print(
    "Golden Records nécessitant une vérification :",
    fiches_a_verifier
)

print(
    "Taux sans anomalie détectée :",
    round(taux_qualite, 2), "%"
)

BILAN FINAL DU DÉDOUBLONNAGE CRM
--------------------------------


,Indicateur,Valeur
0,Fiches CRM analysées,3386.00
1,Golden Records créés,1693.00
2,Fiches principales,1693.00
3,Fiches secondaires,1693.00
4,Réduction des fiches redondantes (%),50.00
5,Golden Records sans anomalie détectée,1692.00
6,Golden Records à vérifier,1.00
7,Taux sans anomalie détectée (%),99.94
8,Références CRM invalides,0.00
9,Identifiants CRM dupliqués,0.00



SYNTHÈSE DU PROJET
------------------
Fiches CRM analysées : 3386
Golden Records créés : 1693
Fiches secondaires identifiées : 1693
Réduction des fiches redondantes : 50.0 %
Golden Records sans anomalie détectée : 1692
Golden Records nécessitant une vérification : 1
Taux sans anomalie détectée : 99.94 %


In [86]:
# Exportation du bilan final du dédoublonnage CRM

from pathlib import Path
import pandas as pd

# 1. Créer le dossier d'exportation
dossier_export = Path("exports")
dossier_export.mkdir(exist_ok=True)

# 2. Définir le chemin du fichier
chemin_bilan = dossier_export / "bilan_final_dedoublonnage_crm.csv"

# 3. Exporter le bilan
bilan_final.to_csv(
    chemin_bilan,
    index=False,
    encoding="utf-8-sig"
)

# 4. Relire le fichier pour vérifier son contenu
df_bilan_export = pd.read_csv(chemin_bilan)

# 5. Vérifier la cohérence
export_bilan_valide = (
    len(df_bilan_export) == len(bilan_final)
    and list(df_bilan_export.columns) == list(bilan_final.columns)
    and df_bilan_export["Indicateur"].equals(
        bilan_final["Indicateur"].reset_index(drop=True)
    )
)

# 6. Afficher les résultats
print("EXPORTATION DU BILAN FINAL CRM")
print("------------------------------")

print("Fichier enregistré :", chemin_bilan)
print("Nombre d'indicateurs :", len(df_bilan_export))
print("Nombre de colonnes :", len(df_bilan_export.columns))

print("\nRÉSULTAT DU CONTRÔLE")
print("--------------------")

if export_bilan_valide:
    print("EXPORT VALIDÉ : bilan enregistré avec succès.")
else:
    print("EXPORT NON VALIDÉ : vérification nécessaire.")

# 7. Afficher le fichier exporté
display(df_bilan_export)

EXPORTATION DU BILAN FINAL CRM
------------------------------
Fichier enregistré : exports\bilan_final_dedoublonnage_crm.csv
Nombre d'indicateurs : 10
Nombre de colonnes : 2

RÉSULTAT DU CONTRÔLE
--------------------
EXPORT VALIDÉ : bilan enregistré avec succès.


,Indicateur,Valeur
0,Fiches CRM analysées,3386.00
1,Golden Records créés,1693.00
2,Fiches principales,1693.00
3,Fiches secondaires,1693.00
4,Réduction des fiches redondantes (%),50.00
5,Golden Records sans anomalie détectée,1692.00
6,Golden Records à vérifier,1.00
7,Taux sans anomalie détectée (%),99.94
8,Références CRM invalides,0.00
9,Identifiants CRM dupliqués,0.00


In [87]:
# Vérification des fichiers destinés à Power BI

from pathlib import Path
import pandas as pd

# 1. Définir les fichiers à vérifier
fichiers_powerbi = {
    "Golden Records": "exports/golden_records_consolides.csv",
    "Correspondances CRM": "exports/correspondance_crm_golden_records.csv",
    "Bilan du dédoublonnage": "exports/bilan_final_dedoublonnage_crm.csv"
}

# 2. Préparer le rapport de vérification
resultats_verification = []

# 3. Vérifier chaque fichier
for nom, chemin in fichiers_powerbi.items():

    fichier = Path(chemin)

    if fichier.exists():

        df = pd.read_csv(fichier)

        resultats_verification.append({
            "Fichier": nom,
            "Statut": "Disponible",
            "Nombre de lignes": len(df),
            "Nombre de colonnes": len(df.columns)
        })

    else:

        resultats_verification.append({
            "Fichier": nom,
            "Statut": "Introuvable",
            "Nombre de lignes": 0,
            "Nombre de colonnes": 0
        })

# 4. Créer le tableau récapitulatif
df_verification_powerbi = pd.DataFrame(
    resultats_verification
)

# 5. Afficher les résultats
print("VÉRIFICATION DES FICHIERS POUR POWER BI")
print("--------------------------------------")

display(df_verification_powerbi)

# 6. Vérifier la disponibilité globale
fichiers_disponibles = (
    df_verification_powerbi["Statut"] == "Disponible"
).all()

if fichiers_disponibles:
    print("\nVALIDATION RÉUSSIE : les trois fichiers sont disponibles.")
else:
    print("\nATTENTION : certains fichiers sont introuvables.")

VÉRIFICATION DES FICHIERS POUR POWER BI
--------------------------------------


,Fichier,Statut,Nombre de lignes,Nombre de colonnes
0,Golden Records,Disponible,1693,33
1,Correspondances CRM,Disponible,3386,4
2,Bilan du dédoublonnage,Disponible,10,2



VALIDATION RÉUSSIE : les trois fichiers sont disponibles.


In [88]:
# Examiner la structure des fichiers CSV

import pandas as pd

# 1. Charger les trois fichiers
df_golden_sql = pd.read_csv(
    "exports/golden_records_consolides.csv",
    dtype={
        "golden_record_id": "string",
        "crm_customer_id": "string"
    }
)

df_mapping_sql = pd.read_csv(
    "exports/correspondance_crm_golden_records.csv",
    dtype={
        "crm_customer_id": "string",
        "golden_record_id": "string"
    }
)

df_bilan_sql = pd.read_csv(
    "exports/bilan_final_dedoublonnage_crm.csv"
)

# 2. Examiner les colonnes et les types de données
for nom, df in [
    ("GOLDEN RECORDS", df_golden_sql),
    ("CORRESPONDANCES CRM", df_mapping_sql),
    ("BILAN DU DÉDOUBLONNAGE", df_bilan_sql)
]:

    print("\n" + "=" * 50)
    print("STRUCTURE :", nom)
    print("=" * 50)

    print("Nombre de lignes :", len(df))
    print("Nombre de colonnes :", len(df.columns))

    structure = pd.DataFrame({
        "Colonne": df.columns,
        "Type_Python": df.dtypes.astype(str).values,
        "Valeurs_manquantes": df.isna().sum().values
    })

    display(structure)


STRUCTURE : GOLDEN RECORDS
Nombre de lignes : 1693
Nombre de colonnes : 33


,Colonne,Type_Python,Valeurs_manquantes
0,prenom,str,0
1,nom,str,0
2,email,str,0
3,phone,str,0
4,address,str,0
5,city,str,0
6,province,str,0
7,postal_code,str,3
8,date_creation,str,0
9,crm_customer_id,string,0



STRUCTURE : CORRESPONDANCES CRM
Nombre de lignes : 3386
Nombre de colonnes : 4


,Colonne,Type_Python,Valeurs_manquantes
0,crm_customer_id,string,0
1,golden_record_id,string,0
2,type_fiche,str,0
3,score_qualite,int64,0



STRUCTURE : BILAN DU DÉDOUBLONNAGE
Nombre de lignes : 10
Nombre de colonnes : 2


,Colonne,Type_Python,Valeurs_manquantes
0,Indicateur,str,0
1,Valeur,float64,0


In [89]:
# Installer les bibliothèques SQL Server

%pip install sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.


In [90]:
# Vérifier les pilotes ODBC

import pyodbc

pilotes = pyodbc.drivers()

print("PILOTES ODBC DISPONIBLES")
print("------------------------")

for pilote in pilotes:
    print(pilote)

PILOTES ODBC DISPONIBLES
------------------------
SQL Server
Oracle dans OraDB21Home1
ODBC Driver 17 for SQL Server
ODBC Driver 18 for SQL Server
CR Sybase Wire Protocol ODBC Driver 7.1.6
CR TextFile ODBC Driver 7.1.6
CR SQL Server Wire Protocol ODBC Driver 8.0.2
Microsoft Access Driver (*.mdb, *.accdb)
Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)
Microsoft Access Text Driver (*.txt, *.csv)
Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


In [91]:
# Connexion Python à SQL Server

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# 1. Paramètres de connexion
serveur = r"DESKTOP-HNDJ8UO"
base_donnees = "Customer360_DB"

# 2. Configuration de la connexion
chaine_connexion = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={serveur};"
    f"DATABASE={base_donnees};"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# 3. Créer le moteur SQLAlchemy
url_connexion = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": chaine_connexion}
)

engine = create_engine(url_connexion)

# 4. Tester la connexion
with engine.connect() as connexion:
    resultat = connexion.execute(
        text("SELECT DB_NAME()")
    ).scalar()

# 5. Afficher les résultats
print("CONNEXION PYTHON À SQL SERVER")
print("-----------------------------")
print("Serveur :", serveur)
print("Base de données :", resultat)
print("Connexion réussie !")

CONNEXION PYTHON À SQL SERVER
-----------------------------
Serveur : DESKTOP-HNDJ8UO
Base de données : Customer360_DB
Connexion réussie !


In [92]:
# Vérification des tables SQL Server

from sqlalchemy import text
import pandas as pd

# 1. Requête pour récupérer les tables
requete = """
SELECT
    TABLE_SCHEMA,
    TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
"""

# 2. Exécuter la requête
with engine.connect() as connexion:
    df_tables = pd.read_sql(
        text(requete),
        connexion
    )

# 3. Afficher les résultats
print("VÉRIFICATION DES TABLES SQL SERVER")
print("----------------------------------")

display(df_tables)

# 4. Vérifier les trois tables attendues
tables_attendues = {
    "Golden_Records",
    "Correspondances_CRM",
    "Bilan_Dedoublonnage"
}

tables_existantes = set(df_tables["TABLE_NAME"])

if tables_attendues.issubset(tables_existantes):
    print("\nVALIDATION RÉUSSIE : les trois tables existent.")
else:
    print("\nATTENTION : certaines tables sont manquantes.")
    print("Tables manquantes :", tables_attendues - tables_existantes)

VÉRIFICATION DES TABLES SQL SERVER
----------------------------------


,TABLE_SCHEMA,TABLE_NAME
0,dbo,Bilan_Dedoublonnage
1,dbo,Correspondances_CRM
2,dbo,Golden_Records



VALIDATION RÉUSSIE : les trois tables existent.


In [93]:
# Vérification des fichiers CSV avant importation

from pathlib import Path
from sqlalchemy import text
import pandas as pd

# 1. Définir les fichiers à importer
fichiers_import = {
    "Golden_Records": "exports/golden_records_consolides.csv",
    "Correspondances_CRM": "exports/correspondance_crm_golden_records.csv",
    "Bilan_Dedoublonnage": "exports/bilan_final_dedoublonnage_crm.csv"
}

# 2. Définir les nombres de lignes attendus
lignes_attendues = {
    "Golden_Records": 1693,
    "Correspondances_CRM": 3386,
    "Bilan_Dedoublonnage": 10
}

# 3. Préparer les résultats
resultats = []

# 4. Vérifier chaque fichier
for table, chemin in fichiers_import.items():

    fichier = Path(chemin)

    if fichier.exists():

        df = pd.read_csv(fichier)

        # Récupérer les colonnes SQL
        requete = text("""
            SELECT COLUMN_NAME
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'dbo'
              AND TABLE_NAME = :table
            ORDER BY ORDINAL_POSITION
        """)

        with engine.connect() as connexion:
            colonnes_sql = [
                ligne[0]
                for ligne in connexion.execute(
                    requete, {"table": table}
                )
            ]

        colonnes_csv = list(df.columns)

        # Vérifier la correspondance des colonnes
        colonnes_identiques = (
            set(colonnes_csv) == set(colonnes_sql)
        )

        # Vérifier si la table SQL est vide
        with engine.connect() as connexion:
            lignes_sql = connexion.execute(
                text(f"SELECT COUNT(*) FROM dbo.{table}")
            ).scalar()

        resultats.append({
            "Table": table,
            "Fichier disponible": True,
            "Lignes CSV": len(df),
            "Lignes attendues": lignes_attendues[table],
            "Colonnes CSV": len(colonnes_csv),
            "Colonnes SQL": len(colonnes_sql),
            "Colonnes compatibles": colonnes_identiques,
            "Lignes déjà dans SQL": lignes_sql,
            "Colonnes manquantes dans SQL":
                ", ".join(sorted(set(colonnes_csv) - set(colonnes_sql))),
            "Colonnes supplémentaires dans SQL":
                ", ".join(sorted(set(colonnes_sql) - set(colonnes_csv)))
        })

    else:

        resultats.append({
            "Table": table,
            "Fichier disponible": False,
            "Lignes CSV": 0,
            "Lignes attendues": lignes_attendues[table],
            "Colonnes CSV": 0,
            "Colonnes SQL": 0,
            "Colonnes compatibles": False,
            "Lignes déjà dans SQL": None,
            "Colonnes manquantes dans SQL": "",
            "Colonnes supplémentaires dans SQL": ""
        })

# 5. Afficher le rapport
df_verification_import = pd.DataFrame(resultats)

print("AUDIT AVANT IMPORTATION SQL SERVER")
print("----------------------------------")

display(df_verification_import)

# 6. Vérifier si les fichiers sont prêts
pret_importation = (
    df_verification_import["Fichier disponible"].all()
    and (
        df_verification_import["Lignes CSV"]
        == df_verification_import["Lignes attendues"]
    ).all()
    and df_verification_import["Colonnes compatibles"].all()
    and (df_verification_import["Lignes déjà dans SQL"] == 0).all()
)

print("\nRÉSULTAT DU CONTRÔLE")
print("--------------------")

if pret_importation:
    print("VALIDATION RÉUSSIE : fichiers prêts pour l'importation.")
else:
    print("ATTENTION : des vérifications supplémentaires sont nécessaires.")

AUDIT AVANT IMPORTATION SQL SERVER
----------------------------------


,Table,Fichier disponible,Lignes CSV,Lignes attendues,Colonnes CSV,Colonnes SQL,Colonnes compatibles,Lignes déjà dans SQL,Colonnes manquantes dans SQL,Colonnes supplémentaires dans SQL
0,Golden_Records,True,1693,1693,33,33,True,0,,
1,Correspondances_CRM,True,3386,3386,4,4,True,0,,
2,Bilan_Dedoublonnage,True,10,10,2,2,True,0,,



RÉSULTAT DU CONTRÔLE
--------------------
VALIDATION RÉUSSIE : fichiers prêts pour l'importation.


In [94]:
# Vérifier les types de données avant importation

import pandas as pd
from sqlalchemy import text

requete_types = text("""
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH,
    IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'dbo'
AND TABLE_NAME IN (
    'Golden_Records',
    'Correspondances_CRM',
    'Bilan_Dedoublonnage'
)
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

with engine.connect() as connexion:
    df_types_sql = pd.read_sql(
        requete_types,
        connexion
    )

print("STRUCTURE DES COLONNES SQL SERVER")
print("---------------------------------")

display(df_types_sql)

STRUCTURE DES COLONNES SQL SERVER
---------------------------------


,TABLE_NAME,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,Bilan_Dedoublonnage,Indicateur,nvarchar,255.0,NO
1,Bilan_Dedoublonnage,Valeur,float,NaN,YES
2,Correspondances_CRM,crm_customer_id,nvarchar,50.0,NO
3,Correspondances_CRM,golden_record_id,nvarchar,50.0,NO
4,Correspondances_CRM,type_fiche,nvarchar,30.0,YES
5,Correspondances_CRM,score_qualite,int,NaN,YES
6,Golden_Records,golden_record_id,nvarchar,50.0,NO
7,Golden_Records,crm_customer_id,nvarchar,50.0,YES
8,Golden_Records,prenom,nvarchar,100.0,YES
9,Golden_Records,nom,nvarchar,100.0,YES


In [95]:
# Préparation des données pour SQL Server

import pandas as pd

# 1. Créer des copies de travail
df_golden_pret = df_golden_sql.copy()
df_mapping_pret = df_mapping_sql.copy()
df_bilan_pret = df_bilan_sql.copy()

# 2. Identifier les colonnes SQL de type date
colonnes_dates = df_types_sql.loc[
    (df_types_sql["TABLE_NAME"] == "Golden_Records")
    & (df_types_sql["DATA_TYPE"] == "date"),
    "COLUMN_NAME"
].tolist()

# 3. Convertir les dates
for colonne in colonnes_dates:
    df_golden_pret[colonne] = pd.to_datetime(
        df_golden_pret[colonne],
        errors="coerce"
    )

# 4. Vérifier les dates manquantes
print("VÉRIFICATION DES DATES")
print("----------------------")

display(
    df_golden_pret[colonnes_dates].isna().sum()
)

# 5. Vérifier les longueurs des textes
tables_pretes = {
    "Golden_Records": df_golden_pret,
    "Correspondances_CRM": df_mapping_pret,
    "Bilan_Dedoublonnage": df_bilan_pret
}

anomalies_longueur = []

for nom_table, df in tables_pretes.items():

    colonnes_texte = df_types_sql.loc[
        (df_types_sql["TABLE_NAME"] == nom_table)
        & (df_types_sql["DATA_TYPE"].isin(
            ["nvarchar", "varchar", "char", "nchar"]
        )),
        ["COLUMN_NAME", "CHARACTER_MAXIMUM_LENGTH"]
    ]

    for _, ligne in colonnes_texte.iterrows():

        colonne = ligne["COLUMN_NAME"]
        longueur_max = ligne["CHARACTER_MAXIMUM_LENGTH"]

        if pd.notna(longueur_max) and longueur_max > 0:

            nombre_anomalies = (
                df[colonne]
                .dropna()
                .astype(str)
                .str.len()
                .gt(int(longueur_max))
                .sum()
            )

            if nombre_anomalies > 0:
                anomalies_longueur.append({
                    "Table": nom_table,
                    "Colonne": colonne,
                    "Longueur maximale": int(longueur_max),
                    "Valeurs trop longues": int(nombre_anomalies)
                })

# 6. Afficher les résultats
print("\nCONTRÔLE DES LONGUEURS")
print("----------------------")

if anomalies_longueur:
    display(pd.DataFrame(anomalies_longueur))
else:
    print("Aucune valeur texte ne dépasse la longueur SQL.")

# 7. Vérifier les volumes
print("\nDONNÉES PRÉPARÉES")
print("----------------------")

print("Golden Records :", len(df_golden_pret))
print("Correspondances CRM :", len(df_mapping_pret))
print("Bilan du dédoublonnage :", len(df_bilan_pret))

VÉRIFICATION DES DATES
----------------------


date_creation           0
date_creation_fiable    1
dtype: int64


CONTRÔLE DES LONGUEURS
----------------------
Aucune valeur texte ne dépasse la longueur SQL.

DONNÉES PRÉPARÉES
----------------------
Golden Records : 1693
Correspondances CRM : 3386
Bilan du dédoublonnage : 10


In [96]:
# Importation des Golden Records dans SQL Server

import pandas as pd
from sqlalchemy import text

# 1. Préparer les données
df_import_golden = df_golden_pret.copy()

# Convertir les valeurs manquantes en NULL SQL
df_import_golden = df_import_golden.astype(object).where(
    pd.notna(df_import_golden),
    None
)

# 2. Vérifier que la table SQL est vide
with engine.connect() as connexion:
    lignes_avant = connexion.execute(
        text("SELECT COUNT(*) FROM dbo.Golden_Records")
    ).scalar()

print("Lignes présentes avant importation :", lignes_avant)

# 3. Importer uniquement si la table est vide
if lignes_avant == 0:

    df_import_golden.to_sql(
        name="Golden_Records",
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=500
    )

    print("Importation terminée.")

else:
    print("Importation annulée : la table contient déjà des données.")

# 4. Vérifier le nombre de lignes dans SQL Server
with engine.connect() as connexion:

    lignes_apres = connexion.execute(
        text("SELECT COUNT(*) FROM dbo.Golden_Records")
    ).scalar()

print("\nVÉRIFICATION DE L'IMPORTATION")
print("-----------------------------")
print("Lignes attendues :", len(df_import_golden))
print("Lignes présentes dans SQL Server :", lignes_apres)

if lignes_apres == len(df_import_golden):
    print("VALIDATION RÉUSSIE : 1693 Golden Records présents.")
else:
    print("ATTENTION : vérifier les données importées.")

Lignes présentes avant importation : 0
Importation terminée.

VÉRIFICATION DE L'IMPORTATION
-----------------------------
Lignes attendues : 1693
Lignes présentes dans SQL Server : 1693
VALIDATION RÉUSSIE : 1693 Golden Records présents.


In [97]:
# Importation des correspondances CRM

import pandas as pd
from sqlalchemy import text

# 1. Préparer les données
df_import_mapping = df_mapping_pret.copy()

# Convertir les valeurs manquantes en NULL SQL
df_import_mapping = df_import_mapping.astype(object).where(
    pd.notna(df_import_mapping),
    None
)

# 2. Vérifier que la table SQL est vide
with engine.connect() as connexion:
    lignes_avant = connexion.execute(
        text("SELECT COUNT(*) FROM dbo.Correspondances_CRM")
    ).scalar()

print("Lignes présentes avant importation :", lignes_avant)

# 3. Importer uniquement si la table est vide
if lignes_avant == 0:

    df_import_mapping.to_sql(
        name="Correspondances_CRM",
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=500
    )

    print("Importation terminée.")

else:
    print("Importation annulée : la table contient déjà des données.")

# 4. Vérifier les données importées
with engine.connect() as connexion:

    lignes_apres = connexion.execute(
        text("SELECT COUNT(*) FROM dbo.Correspondances_CRM")
    ).scalar()

    identifiants_uniques = connexion.execute(
        text("""
            SELECT COUNT(DISTINCT crm_customer_id)
            FROM dbo.Correspondances_CRM
        """)
    ).scalar()

    golden_references = connexion.execute(
        text("""
            SELECT COUNT(DISTINCT golden_record_id)
            FROM dbo.Correspondances_CRM
        """)
    ).scalar()

# 5. Afficher les résultats
print("\nVÉRIFICATION DE L'IMPORTATION")
print("-----------------------------")

print("Lignes attendues :", len(df_import_mapping))
print("Lignes présentes dans SQL Server :", lignes_apres)
print("Identifiants CRM uniques :", identifiants_uniques)
print("Golden Records référencés :", golden_references)

# 6. Validation
if (
    lignes_apres == 3386
    and identifiants_uniques == 3386
    and golden_references == 1693
):
    print("\nVALIDATION RÉUSSIE : correspondances CRM importées.")
else:
    print("\nATTENTION : vérification supplémentaire nécessaire.")

Lignes présentes avant importation : 0
Importation terminée.

VÉRIFICATION DE L'IMPORTATION
-----------------------------
Lignes attendues : 3386
Lignes présentes dans SQL Server : 3386
Identifiants CRM uniques : 3386
Golden Records référencés : 1693

VALIDATION RÉUSSIE : correspondances CRM importées.


In [98]:
# Importation du bilan du dédoublonnage

import pandas as pd
from sqlalchemy import text

# 1. Préparer les données
df_import_bilan = df_bilan_pret.copy()

# 2. Vérifier que la table SQL est vide
with engine.connect() as connexion:

    lignes_avant = connexion.execute(
        text("SELECT COUNT(*) FROM dbo.Bilan_Dedoublonnage")
    ).scalar()

print("Lignes présentes avant importation :", lignes_avant)

# 3. Importer uniquement si la table est vide
if lignes_avant == 0:

    df_import_bilan.to_sql(
        name="Bilan_Dedoublonnage",
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False
    )

    print("Importation terminée.")

else:
    print("Importation annulée : la table contient déjà des données.")

# 4. Vérifier les données importées
with engine.connect() as connexion:

    df_bilan_sql_server = pd.read_sql(
        text("""
            SELECT Indicateur, Valeur
            FROM dbo.Bilan_Dedoublonnage
        """),
        connexion
    )

# 5. Afficher les résultats
print("\nVÉRIFICATION DE L'IMPORTATION")
print("-----------------------------")

print("Indicateurs attendus :", len(df_import_bilan))

print(
    "Indicateurs présents dans SQL Server :",
    len(df_bilan_sql_server)
)

print(
    "Indicateurs uniques :",
    df_bilan_sql_server["Indicateur"].nunique()
)

# 6. Vérifier la conformité
bilan_valide = (
    len(df_bilan_sql_server) == 10
    and df_bilan_sql_server["Indicateur"].is_unique
    and df_bilan_sql_server["Indicateur"].notna().all()
    and df_bilan_sql_server["Valeur"].notna().all()
)

if bilan_valide:
    print("\nVALIDATION RÉUSSIE : bilan importé dans SQL Server.")
else:
    print("\nATTENTION : vérification supplémentaire nécessaire.")

# 7. Afficher le bilan
display(df_bilan_sql_server)

Lignes présentes avant importation : 0
Importation terminée.

VÉRIFICATION DE L'IMPORTATION
-----------------------------
Indicateurs attendus : 10
Indicateurs présents dans SQL Server : 10
Indicateurs uniques : 10

VALIDATION RÉUSSIE : bilan importé dans SQL Server.


,Indicateur,Valeur
0,Fiches CRM analysées,3386.00
1,Fiches principales,1693.00
2,Fiches secondaires,1693.00
3,Golden Records à vérifier,1.00
4,Golden Records créés,1693.00
5,Golden Records sans anomalie détectée,1692.00
6,Identifiants CRM dupliqués,0.00
7,Réduction des fiches redondantes (%),50.00
8,Références CRM invalides,0.00
9,Taux sans anomalie détectée (%),99.94


In [99]:
# Audit final de l'intégrité SQL Server

import pandas as pd
from sqlalchemy import text

# 1. Définir les contrôles SQL
controles_sql = {
    "Golden Records": """
        SELECT COUNT(*)
        FROM dbo.Golden_Records
    """,

    "Correspondances CRM": """
        SELECT COUNT(*)
        FROM dbo.Correspondances_CRM
    """,

    "Indicateurs du bilan": """
        SELECT COUNT(*)
        FROM dbo.Bilan_Dedoublonnage
    """,

    "Identifiants Golden uniques": """
        SELECT COUNT(DISTINCT golden_record_id)
        FROM dbo.Golden_Records
    """,

    "Identifiants CRM uniques": """
        SELECT COUNT(DISTINCT crm_customer_id)
        FROM dbo.Correspondances_CRM
    """,

    "Références CRM invalides": """
        SELECT COUNT(*)
        FROM dbo.Correspondances_CRM AS c
        LEFT JOIN dbo.Golden_Records AS g
            ON c.golden_record_id = g.golden_record_id
        WHERE g.golden_record_id IS NULL
    """,

    "Golden Records sans fiche principale unique": """
        SELECT COUNT(*)
        FROM dbo.Golden_Records AS g
        LEFT JOIN dbo.Correspondances_CRM AS c
            ON g.golden_record_id = c.golden_record_id
            AND c.type_fiche = N'Principale'
        GROUP BY g.golden_record_id
        HAVING COUNT(c.crm_customer_id) <> 1
    """
}

# 2. Exécuter les contrôles
resultats_audit = []

with engine.connect() as connexion:

    for nom, requete in controles_sql.items():

        if nom == "Golden Records sans fiche principale unique":

            resultat = connexion.execute(
                text(requete)
            ).fetchall()

            valeur = len(resultat)

        else:

            valeur = connexion.execute(
                text(requete)
            ).scalar()

        resultats_audit.append({
            "Contrôle": nom,
            "Résultat": valeur
        })

# 3. Construire le tableau
df_audit_sql = pd.DataFrame(resultats_audit)

# 4. Afficher les résultats
print("AUDIT FINAL DE L'INTÉGRITÉ SQL SERVER")
print("------------------------------------")

display(df_audit_sql)

# 5. Vérifier la conformité
valeurs_attendues = {
    "Golden Records": 1693,
    "Correspondances CRM": 3386,
    "Indicateurs du bilan": 10,
    "Identifiants Golden uniques": 1693,
    "Identifiants CRM uniques": 3386,
    "Références CRM invalides": 0,
    "Golden Records sans fiche principale unique": 0
}

df_audit_sql["Valeur attendue"] = (
    df_audit_sql["Contrôle"].map(valeurs_attendues)
)

df_audit_sql["Validation"] = (
    df_audit_sql["Résultat"]
    == df_audit_sql["Valeur attendue"]
)

print("\nRÉSULTAT DES CONTRÔLES")
print("----------------------")

display(df_audit_sql)

if df_audit_sql["Validation"].all():
    print("\nAUDIT VALIDÉ : intégrité des données confirmée.")
else:
    print("\nATTENTION : certaines anomalies nécessitent une vérification.")

AUDIT FINAL DE L'INTÉGRITÉ SQL SERVER
------------------------------------


,Contrôle,Résultat
0,Golden Records,1693
1,Correspondances CRM,3386
2,Indicateurs du bilan,10
3,Identifiants Golden uniques,1693
4,Identifiants CRM uniques,3386
5,Références CRM invalides,0
6,Golden Records sans fiche principale unique,0



RÉSULTAT DES CONTRÔLES
----------------------


,Contrôle,Résultat,Valeur attendue,Validation
0,Golden Records,1693,1693,True
1,Correspondances CRM,3386,3386,True
2,Indicateurs du bilan,10,10,True
3,Identifiants Golden uniques,1693,1693,True
4,Identifiants CRM uniques,3386,3386,True
5,Références CRM invalides,0,0,True
6,Golden Records sans fiche principale unique,0,0,True



AUDIT VALIDÉ : intégrité des données confirmée.
